# DivGen on GenEval50 under a shared generation protocol

We compare DivGen starting from pink noise (alpha = 0.2 by default) and same-phase whitefloor (alpha = 0.9, gamma = 0.05). The 50 prompts, initial Gaussian samples, noise normalisation and generation settings are shared with FloorRefine: SDXL-Turbo, 512 x 512, one or four denoising steps and four images per prompt. Use one step for the main experiment and four steps for an additional comparison, matching the selected FloorRefine schedule.

Generation uses FP16 UNet inference, FP32 VAE decoding and independent, fixed CPU random streams for each image. Prompt embeddings are prepared once per group. The generator processes images individually while retaining their computation graphs, so the diversity objective still evaluates all four outputs together. The initial noise is the only optimised parameter.

DivGen's official trainer, HPS loss, DINO-DPP objective, regularisation, SGD and candidate selection are retained. Its HPS reduction uses the first image's score for a four-image group with one text embedding. We preserve that behaviour explicitly. The weights are 3 for HPS, 150 for DPP and 0.01 for regularisation; SGD uses learning rate 6, momentum 0.9 and gradient clipping at 0.1. We evaluate 150 candidates with 149 updates and record an intermediate budget after 10 updates.

This evaluates DivGen under our shared generation protocol, rather than reproducing the paper's default configuration. The official SDXL configuration uses one denoising step and pink alpha = 0.2. Here, sampling and decoding follow FloorRefine, while DivGen retains its own optimisation objective and FP16 noise parameter. No staged VJP, gradient scaling or retry is used.

Place this notebook in `denoising_optimization/`. Results are saved under `outputs/divgen_geneval50/`. The selected upstream source is included with its licence and recorded hashes; no repository clone or package upgrade is required.

| Sections | Kernel | Purpose |
| --- | --- | --- |
| 1-6 | Generation | Generate the two initialisations and record optimisation cost |
| 7-14 | Metrics | Evaluate saved outputs and compare quality, diversity and cost |

MANIQA runs automatically in `.venv-maniqa`. Section 7 checks that generation is complete before evaluating saved images.


## 1. Prepare the generation environment

Use the existing generation environment. This notebook needs PyTorch, torchvision, Diffusers, Transformers, Accelerate, HPSv2, timm, NumPy, Pillow and pandas. It does not import optional DivGen rewards or unrelated generator pipelines. The existing tokenizer preparation runs below. Package versions and GPU information are recorded with the experiment.

The HPS constructor may download its OpenCLIP backbone as well as the HPS checkpoint. A full four-image backward pass requires substantial GPU memory. UNet calls and VAE decoding use activation checkpointing. Run one prompt before committing to all 50; the notebook does not silently change precision or the optimisation method.


In [ ]:
from importlib.metadata import distribution
from pathlib import Path

path = Path(distribution("hpsv2").locate_file(
    "hpsv2/src/open_clip/factory.py"
))
source = path.read_text(encoding="utf-8")
bad_line = "from turtle import forward"

if bad_line in source:
    backup = path.with_suffix(".py.bak")
    if not backup.exists():
        backup.write_text(source, encoding="utf-8")
    source = "\n".join(
        line for line in source.split("\n")
        if line.strip() != bad_line
    )
    path.write_text(source, encoding="utf-8")
    print("Removed the unused turtle import from HPSv2.")
else:
    print("The turtle import is absent; inspect the full traceback if an error persists.")

from importlib.metadata import distribution
from pathlib import Path
import urllib.request
import hashlib

target = Path(distribution("hpsv2").locate_file(
    "hpsv2/src/open_clip/bpe_simple_vocab_16e6.txt.gz"
))

url = (
    "https://raw.githubusercontent.com/openai/CLIP/main/"
    "clip/bpe_simple_vocab_16e6.txt.gz"
)
expected_sha = "924691ac288e54409236115652ad4aa250f48203de50a9e4722a6ecd48d6804a"

if not target.exists():
    with urllib.request.urlopen(url, timeout=60) as response:
        data = response.read()
    assert hashlib.sha256(data).hexdigest() == expected_sha, "File checksum verification failed."
    target.write_bytes(data)

assert hashlib.sha256(target.read_bytes()).hexdigest() == expected_sha
print("HPSv2 vocabulary ready:", target)


## 2. Select prompts and settings

We use the same fixed 50-prompt subset as FloorRefine, selected from the 400-prompt GenEval pool. Category counts use proportional allocation with the largest-remainder method, followed by seeded hash ranking within each category (`selection_seed = 20261023`). The subset contains 7 single-object, 9 two-object, 7 counting, 9 colour, 9 position and 9 colour-attribute-binding prompts.

Set `PROMPT_LIMIT=1` for a complete one-prompt trial of both initialisations. Restore `None` for all 50 prompts. Keep `RUN_NAME=None` to separate these runs automatically.

`SAVE_BUDGETS=[21,150]` records selected candidates after at most 20 and 149 updates from the same trajectory. Each group also includes its unoptimised baseline. The full experiment saves 1,200 images. The 21-evaluation timing is a trajectory prefix, not an independently timed short run.

Set `NUM_STEPS=1` or `NUM_STEPS=4` to switch denoising schedules. With `RUN_NAME=None`, the run name includes the schedule and current pink alpha. After changing the setting, rerun the configuration and generation cells, or run Sections 1–6 from a fresh generation kernel. The optimisation budget, rewards, precision and noise parameters remain unchanged. The default `PROMPT_LIMIT=1` runs a trial; set it to `None` for the formal 50-prompt experiment.


In [ ]:
import gc, hashlib, importlib.metadata, json, math, os, sys, time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

PROJECT_ROOT = None  # Set an absolute project path if automatic detection fails.
def find_project_root(value=None):
    cwd = Path.cwd().resolve()
    candidates = [Path(value).expanduser().resolve()] if value else [cwd, *cwd.parents]
    for root in candidates:
        if (root/'noise_init/metric_runner.py').is_file() and (root/'pc_specific_psd/compat_metrics.py').is_file():
            return root
    raise FileNotFoundError('Set PROJECT_ROOT to the project root containing noise_init and pc_specific_psd.')

PROJECT_ROOT = find_project_root(PROJECT_ROOT)
MODEL_ID = 'stabilityai/sdxl-turbo'
VAE_ID = 'madebyollin/sdxl-vae-fp16-fix'
DINO_ID = 'facebook/dinov2-base'
HPS_REPO, HPS_FILE = 'xswu/HPSv2', 'HPS_v2.1_compressed.pt'
CACHE_DIR = str(PROJECT_ROOT/'noise_init/cache')
HEIGHT = WIDTH = 512
NUM_STEPS = 1  # Select 1 (main experiment) or 4 (additional comparison).
NUM_IMAGES = 4
N_EVALUATIONS = 150  # Includes the initial candidate: 149 SGD updates, as in the official loop.
SAVE_BUDGETS = [21, 150]  # Candidates after at most 10 and 149 updates, from one trajectory.
LR, MOMENTUM, GRAD_CLIP = 6.0, 0.9, 0.1
PINK_ALPHA = 0.2
WHITEFLOOR_ALPHA, WHITEFLOOR_GAMMA = 0.9, 0.05
INIT_CONFIGS = {
    'pink': {'method': 'pink', 'alpha': PINK_ALPHA, 'gamma': None,
             'normalization': 'per_sample_per_channel_zero_mean_unit_std', 'label': f'Pink (alpha={PINK_ALPHA:g})'},
    'whitefloor': {'method': 'same_phase_whitefloor', 'alpha': WHITEFLOOR_ALPHA,
                  'gamma': WHITEFLOOR_GAMMA,
                  'normalization': 'per_sample_per_channel_zero_mean_unit_std',
                  'label': f'Whitefloor (alpha={WHITEFLOOR_ALPHA:g}, gamma={WHITEFLOOR_GAMMA:g})'},
}
HPS_WEIGHT, DPP_WEIGHT, REG_WEIGHT = 3.0, 150.0, 0.01
HPS_REDUCTION = 'official_diagonal'  # The reduction implemented by the official HPSLoss.
SCHEDULER_SEED = 42  # Upstream trainer argument; the shared generator uses the per-image seeds below.
ROLLOUT_SEED_OFFSET = 200000011
MODEL_DTYPE = torch.float16
GRAD_CHECKPOINT = True
NORMALIZATION_PROFILE = 'per_sample_per_channel_zero_mean_unit_std'
DEVICE, DTYPE = 'cuda', torch.float16
PROMPT_LIMIT = None  # None = all 50. Set 1 for a separate smoke run before the full batch.
RESUME = True  # Reuse completed prompt groups; interrupted groups restart from their initial noise.
LOG_EVERY = 10
RUN_NAME = None  # Automatic name includes denoising steps, pink alpha, budget and prompt count.
OUTPUT_ROOT = PROJECT_ROOT/'denoising_optimization/outputs/divgen_geneval50'
DIVGEN_REVISION = '6166a2d4c1fb06dd6984adcd18a3261df7030e74'
ADAPTER_VERSION = 'shared_generation_divgen_v1'

def file_sha(path):
    with Path(path).open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()

def object_sha(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':')).encode()).hexdigest()

def atomic_json(path, data):
    path = Path(path)
    temp = path.with_suffix(path.suffix+'.tmp')
    temp.write_text(json.dumps(data, indent=2, allow_nan=False), encoding='utf-8')
    temp.replace(path)

def sync_time():
    if torch.cuda.is_available(): torch.cuda.synchronize()
    return time.perf_counter()

def memory_snapshot():
    result = {}
    for name in ['memory.current', 'memory.max', 'memory.peak', 'memory.events']:
        p = Path('/sys/fs/cgroup')/name
        if p.exists(): result[name] = p.read_text().strip()
    if torch.cuda.is_available():
        result.update(cuda_allocated_gib=torch.cuda.memory_allocated()/2**30,
                      cuda_reserved_gib=torch.cuda.memory_reserved()/2**30)
    return result

def release_models():
    for name in ['pipe', 'generation_adapter', 'hps_reward', 'dino_backend', 'dpp_objective']:
        globals().pop(name, None)
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    try:
        import ctypes
        ctypes.CDLL('libc.so.6').malloc_trim(0)
    except (OSError, AttributeError):
        pass

def experiment_config(prompts):
    if list(INIT_CONFIGS) != ['pink', 'whitefloor']:
        raise ValueError('This paired experiment requires pink and whitefloor, in that order.')
    source = PROJECT_ROOT/'noise_init/noise_methods.py'
    if not source.is_file() or not (PROJECT_ROOT/'pc_specific_psd/_compat_common.py').is_file():
        raise FileNotFoundError('The project noise module and compatibility loader are required for whitefloor.')
    if HPS_REDUCTION != 'official_diagonal':
        raise ValueError('This experiment uses the unchanged official HPS reduction.')
    if N_EVALUATIONS < 2 or sorted(set(SAVE_BUDGETS)) != SAVE_BUDGETS or not SAVE_BUDGETS:
        raise ValueError('Use at least two evaluations and sorted, unique SAVE_BUDGETS.')
    if SAVE_BUDGETS[-1] != N_EVALUATIONS or SAVE_BUDGETS[0] < 2:
        raise ValueError('SAVE_BUDGETS must end at N_EVALUATIONS and start at two or more.')
    if type(NUM_STEPS) is not int or NUM_STEPS not in (1, 4):
        raise ValueError('NUM_STEPS must be 1 or 4.')
    if (HEIGHT, WIDTH, NUM_IMAGES) != (512, 512, 4):
        raise ValueError('This experiment fixes 512 x 512 and four images per prompt.')
    return dict(adapter_version=ADAPTER_VERSION, upstream_revision=DIVGEN_REVISION,
        model_id=MODEL_ID, vae_id=VAE_ID, dino_id=DINO_ID, hps_repo=HPS_REPO, hps_file=HPS_FILE,
        height=HEIGHT, width=WIDTH, num_inference_steps=NUM_STEPS, num_images=NUM_IMAGES,
        scheduler='EulerAncestralDiscreteScheduler', timestep_spacing='trailing', guidance_scale=0.0,
        dtype=str(DTYPE), n_evaluations=N_EVALUATIONS, n_updates=N_EVALUATIONS-1,
        save_budgets=SAVE_BUDGETS, lr=LR, momentum=MOMENTUM, grad_clip=GRAD_CLIP,
        initializations=INIT_CONFIGS, noise_methods_sha256=file_sha(source),
        precision_policy=dict(initial_noise_construction='torch.float32',
                              optimized_noise=str(DTYPE), unet=str(MODEL_DTYPE), vae='torch.float32',
                              hps='official mixed FP16/FP32 weights', dino='torch.float32'),
        hps_weight=HPS_WEIGHT, dpp_weight=DPP_WEIGHT, reg_weight=REG_WEIGHT,
        hps_reduction=HPS_REDUCTION, rollout_seed_offset=ROLLOUT_SEED_OFFSET,
        scheduler_rng='independent CPU generator per image, reset per candidate',
        generation_microbatch=1, gradient_checkpointing=GRAD_CHECKPOINT,
        normalization=NORMALIZATION_PROFILE, adapter_sources=SHARED_SOURCE_HASHES,
        execution='official LatentNoiseTrainer.train with shared FloorRefine generation',
        source_files=OFFICIAL_SOURCE_HASHES,
        prompts=prompts)

def generation_run_name(prompt_count):
    if type(NUM_STEPS) is not int or NUM_STEPS not in (1, 4):
        raise ValueError('NUM_STEPS must be 1 or 4.')
    return RUN_NAME or (f'geneval{prompt_count}_divgen_shared_sdxl{NUM_STEPS}step_'
                        f'pink{PINK_ALPHA:g}_whitefloor_eval{N_EVALUATIONS}_v1')

print('Denoising steps:', NUM_STEPS, '| Candidate evaluations:', N_EVALUATIONS)
print('Pink alpha:', PINK_ALPHA, '| Prompt limit:', PROMPT_LIMIT)
print('Generation run:', generation_run_name(50 if PROMPT_LIMIT is None else PROMPT_LIMIT))


In [ ]:
# Fixed GenEval prompts and sampling settings. Run this cell before generation.
PROMPT_SOURCE = {'selection_seed': 20261023, 'selection': 'proportional categories, largest remainder (ties ascending category), within category SHA256(seed|prompt_id|prompt); restore source order', 'category_counts': {'single_object': 7, 'two_object': 9, 'counting': 7, 'colors': 9, 'position': 9, 'color_attr': 9}}

GENEVAL50 = [
    {'prompt_id': 'geneval_0010', 'prompt': 'a photo of a microwave', 'category': 'single_object', 'batch_seed': 4100100, 'source_row': 11},
    {'prompt_id': 'geneval_0013', 'prompt': 'a photo of a zebra', 'category': 'single_object', 'batch_seed': 4100130, 'source_row': 14},
    {'prompt_id': 'geneval_0014', 'prompt': 'a photo of a parking meter', 'category': 'single_object', 'batch_seed': 4100140, 'source_row': 15},
    {'prompt_id': 'geneval_0047', 'prompt': 'a photo of an oven', 'category': 'single_object', 'batch_seed': 4100470, 'source_row': 48},
    {'prompt_id': 'geneval_0058', 'prompt': 'a photo of a fire hydrant', 'category': 'single_object', 'batch_seed': 4100580, 'source_row': 59},
    {'prompt_id': 'geneval_0059', 'prompt': 'a photo of an airplane', 'category': 'single_object', 'batch_seed': 4100590, 'source_row': 60},
    {'prompt_id': 'geneval_0071', 'prompt': 'a photo of a bus', 'category': 'single_object', 'batch_seed': 4100710, 'source_row': 72},
    {'prompt_id': 'geneval_0081', 'prompt': 'a photo of a toothbrush and a snowboard', 'category': 'two_object', 'batch_seed': 4100810, 'source_row': 82},
    {'prompt_id': 'geneval_0091', 'prompt': 'a photo of a hair drier and a bear', 'category': 'two_object', 'batch_seed': 4100910, 'source_row': 92},
    {'prompt_id': 'geneval_0102', 'prompt': 'a photo of a skateboard and a cake', 'category': 'two_object', 'batch_seed': 4101020, 'source_row': 103},
    {'prompt_id': 'geneval_0124', 'prompt': 'a photo of a scissors and a sandwich', 'category': 'two_object', 'batch_seed': 4101240, 'source_row': 125},
    {'prompt_id': 'geneval_0125', 'prompt': 'a photo of a pizza and a book', 'category': 'two_object', 'batch_seed': 4101250, 'source_row': 126},
    {'prompt_id': 'geneval_0136', 'prompt': 'a photo of a cow and a horse', 'category': 'two_object', 'batch_seed': 4101360, 'source_row': 137},
    {'prompt_id': 'geneval_0160', 'prompt': 'a photo of a tie and a broccoli', 'category': 'two_object', 'batch_seed': 4101600, 'source_row': 161},
    {'prompt_id': 'geneval_0172', 'prompt': 'a photo of a microwave and a bench', 'category': 'two_object', 'batch_seed': 4101720, 'source_row': 173},
    {'prompt_id': 'geneval_0178', 'prompt': 'a photo of a baseball bat and a giraffe', 'category': 'two_object', 'batch_seed': 4101780, 'source_row': 179},
    {'prompt_id': 'geneval_0185', 'prompt': 'a photo of two ties', 'category': 'counting', 'batch_seed': 4101850, 'source_row': 186},
    {'prompt_id': 'geneval_0201', 'prompt': 'a photo of three handbags', 'category': 'counting', 'batch_seed': 4102010, 'source_row': 202},
    {'prompt_id': 'geneval_0234', 'prompt': 'a photo of two sandwichs', 'category': 'counting', 'batch_seed': 4102340, 'source_row': 235},
    {'prompt_id': 'geneval_0244', 'prompt': 'a photo of two teddy bears', 'category': 'counting', 'batch_seed': 4102440, 'source_row': 245},
    {'prompt_id': 'geneval_0245', 'prompt': 'a photo of four skateboards', 'category': 'counting', 'batch_seed': 4102450, 'source_row': 246},
    {'prompt_id': 'geneval_0254', 'prompt': 'a photo of four benchs', 'category': 'counting', 'batch_seed': 4102540, 'source_row': 255},
    {'prompt_id': 'geneval_0256', 'prompt': 'a photo of four frisbees', 'category': 'counting', 'batch_seed': 4102560, 'source_row': 257},
    {'prompt_id': 'geneval_0273', 'prompt': 'a photo of a red zebra', 'category': 'colors', 'batch_seed': 4102730, 'source_row': 274},
    {'prompt_id': 'geneval_0298', 'prompt': 'a photo of a purple carrot', 'category': 'colors', 'batch_seed': 4102980, 'source_row': 299},
    {'prompt_id': 'geneval_0300', 'prompt': 'a photo of a purple potted plant', 'category': 'colors', 'batch_seed': 4103000, 'source_row': 301},
    {'prompt_id': 'geneval_0301', 'prompt': 'a photo of a purple backpack', 'category': 'colors', 'batch_seed': 4103010, 'source_row': 302},
    {'prompt_id': 'geneval_0309', 'prompt': 'a photo of a yellow parking meter', 'category': 'colors', 'batch_seed': 4103090, 'source_row': 310},
    {'prompt_id': 'geneval_0320', 'prompt': 'a photo of a green clock', 'category': 'colors', 'batch_seed': 4103200, 'source_row': 321},
    {'prompt_id': 'geneval_0326', 'prompt': 'a photo of a black hot dog', 'category': 'colors', 'batch_seed': 4103260, 'source_row': 327},
    {'prompt_id': 'geneval_0337', 'prompt': 'a photo of a blue carrot', 'category': 'colors', 'batch_seed': 4103370, 'source_row': 338},
    {'prompt_id': 'geneval_0349', 'prompt': 'a photo of a blue book', 'category': 'colors', 'batch_seed': 4103490, 'source_row': 350},
    {'prompt_id': 'geneval_0368', 'prompt': 'a photo of a baseball glove below an umbrella', 'category': 'position', 'batch_seed': 4103680, 'source_row': 369},
    {'prompt_id': 'geneval_0377', 'prompt': 'a photo of a tv remote left of an umbrella', 'category': 'position', 'batch_seed': 4103770, 'source_row': 378},
    {'prompt_id': 'geneval_0385', 'prompt': 'a photo of a kite above a toothbrush', 'category': 'position', 'batch_seed': 4103850, 'source_row': 386},
    {'prompt_id': 'geneval_0389', 'prompt': 'a photo of a stop sign above a chair', 'category': 'position', 'batch_seed': 4103890, 'source_row': 390},
    {'prompt_id': 'geneval_0398', 'prompt': 'a photo of a toilet left of a kite', 'category': 'position', 'batch_seed': 4103980, 'source_row': 399},
    {'prompt_id': 'geneval_0408', 'prompt': 'a photo of a frisbee below a horse', 'category': 'position', 'batch_seed': 4104080, 'source_row': 409},
    {'prompt_id': 'geneval_0419', 'prompt': 'a photo of a hot dog above a knife', 'category': 'position', 'batch_seed': 4104190, 'source_row': 420},
    {'prompt_id': 'geneval_0421', 'prompt': 'a photo of a chair left of a zebra', 'category': 'position', 'batch_seed': 4104210, 'source_row': 422},
    {'prompt_id': 'geneval_0438', 'prompt': 'a photo of a horse right of a broccoli', 'category': 'position', 'batch_seed': 4104380, 'source_row': 439},
    {'prompt_id': 'geneval_0455', 'prompt': 'a photo of a green skis and a brown airplane', 'category': 'color_attr', 'batch_seed': 4104550, 'source_row': 456},
    {'prompt_id': 'geneval_0485', 'prompt': 'a photo of a pink tv remote and a blue airplane', 'category': 'color_attr', 'batch_seed': 4104850, 'source_row': 486},
    {'prompt_id': 'geneval_0491', 'prompt': 'a photo of a white pizza and a green umbrella', 'category': 'color_attr', 'batch_seed': 4104910, 'source_row': 492},
    {'prompt_id': 'geneval_0500', 'prompt': 'a photo of a yellow dining table and a pink dog', 'category': 'color_attr', 'batch_seed': 4105000, 'source_row': 501},
    {'prompt_id': 'geneval_0505', 'prompt': 'a photo of a red laptop and a brown car', 'category': 'color_attr', 'batch_seed': 4105050, 'source_row': 506},
    {'prompt_id': 'geneval_0509', 'prompt': 'a photo of a brown computer mouse and a purple bottle', 'category': 'color_attr', 'batch_seed': 4105090, 'source_row': 510},
    {'prompt_id': 'geneval_0542', 'prompt': 'a photo of a brown giraffe and a white stop sign', 'category': 'color_attr', 'batch_seed': 4105420, 'source_row': 543},
    {'prompt_id': 'geneval_0545', 'prompt': 'a photo of a red clock and a black cell phone', 'category': 'color_attr', 'batch_seed': 4105450, 'source_row': 546},
    {'prompt_id': 'geneval_0547', 'prompt': 'a photo of a red cup and a pink handbag', 'category': 'color_attr', 'batch_seed': 4105470, 'source_row': 548},
]


## 3. Load the generation protocol and DivGen objectives

The embedded files contain the selected [DivGen](https://github.com/anneharrington/divgen) definitions with unchanged function bodies. Imports for inactive branches are omitted. A generation adapter supplies the FloorRefine sampling protocol to the official trainer. The shared sampling functions are included here, so running this notebook does not require executing the FloorRefine notebook.

Both initialisations call the project's noise functions and population-standard-deviation normalisation. Noise is constructed in FP32 and converted to DivGen's FP16 optimisation parameter. VAE decoding remains FP32. Preprocessed pixels are cast to the HPS encoder's FP16 input type; its score reduction and loss remain unchanged. DINO processes FP32 images.


In [ ]:
OFFICIAL_SOURCES = {'LICENSE': 'MIT License\n\nCopyright (c) 2026 Anne Harrington, A. Sophia Koepke, Shyamgopal Karthik\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'models/utils.py': '"""Utility functions for model initialization and management."""\n\n\ndef freeze_params(params):\n    """Freeze model parameters to prevent gradient updates.\n    \n    Args:\n        params: Iterator of model parameters (e.g., model.parameters())\n    """\n    for param in params:\n        param.requires_grad = False\n\n', 'models/RewardStableDiffusionXL.py': 'from typing import List, Optional, Union\n\nimport torch\nfrom diffusers import AutoencoderKL, StableDiffusionXLPipeline, UNet2DConditionModel\nfrom diffusers.pipelines.stable_diffusion_xl.pipeline_stable_diffusion_xl import (\n    retrieve_timesteps,\n)\nfrom diffusers.schedulers import KarrasDiffusionSchedulers\nfrom transformers import (\n    CLIPImageProcessor,\n    CLIPTextModel,\n    CLIPTextModelWithProjection,\n    CLIPTokenizer,\n    CLIPVisionModelWithProjection,\n)\n\nfrom models.utils import freeze_params\n\n\nclass RewardStableDiffusionXL(StableDiffusionXLPipeline):\n    def __init__(\n        self,\n        vae: AutoencoderKL,\n        text_encoder: CLIPTextModel,\n        text_encoder_2: CLIPTextModelWithProjection,\n        tokenizer: CLIPTokenizer,\n        tokenizer_2: CLIPTokenizer,\n        unet: UNet2DConditionModel,\n        scheduler: KarrasDiffusionSchedulers,\n        image_encoder: CLIPVisionModelWithProjection = None,\n        feature_extractor: CLIPImageProcessor = None,\n        force_zeros_for_empty_prompt: bool = True,\n        add_watermarker: bool = False,\n        is_hyper: bool = False,\n    ):\n        super().__init__(\n            vae,\n            text_encoder,\n            text_encoder_2,\n            tokenizer,\n            tokenizer_2,\n            unet,\n            scheduler,\n            image_encoder,\n            feature_extractor,\n            force_zeros_for_empty_prompt,\n            add_watermarker,\n        )\n        # enable checkpointing\n        self.unet.enable_gradient_checkpointing()\n        self.vae.enable_gradient_checkpointing()\n        self.text_encoder.eval()\n        self.text_encoder_2.eval()\n        self.unet.eval()\n        self.vae.eval()\n        self.is_hyper = is_hyper\n\n        # freeze diffusion parameters\n        freeze_params(self.vae.parameters())\n        freeze_params(self.unet.parameters())\n        freeze_params(self.text_encoder.parameters())\n        freeze_params(self.text_encoder_2.parameters())\n\n    def decode_latents_tensors(self, latents):\n        scaled_latents = latents / self.vae.config.scaling_factor\n        image = self.vae.decode(scaled_latents).sample\n        image = (image / 2 + 0.5).clamp(0, 1)\n        # Clear cache after VAE decode to free memory\n        # Note: VAE is frozen, so intermediate activations can be cleared\n        del scaled_latents\n        torch.cuda.empty_cache()\n        return image\n\n    def apply(\n        self,\n        latents: torch.Tensor,\n        prompt: Union[str, List[str]] = None,\n        prompt_2: Optional[Union[str, List[str]]] = None,\n        height: Optional[int] = None,\n        width: Optional[int] = None,\n        num_inference_steps: int = 1,\n        guidance_scale: float = 0.0,\n        timesteps: List[int] = None,\n        denoising_end: Optional[float] = None,\n        negative_prompt: Optional[Union[str, List[str]]] = None,\n        negative_prompt_2: Optional[Union[str, List[str]]] = None,\n        num_images_per_prompt: Optional[int] = 1,\n        eta: float = 0.0,\n        generator: Optional[Union[torch.Generator, List[torch.Generator]]] = None,\n    ) -> torch.Tensor:\n        if self.is_hyper:\n            timesteps = [800]\n        # 0. Default height and width to unet\n        height = height or self.default_sample_size * self.vae_scale_factor\n        width = width or self.default_sample_size * self.vae_scale_factor\n\n        original_size = (height, width)\n        target_size = (height, width)\n\n        # 1. Check inputs. Raise error if not correct\n        self.check_inputs(\n            prompt,\n            prompt_2,\n            height,\n            width,\n            callback_steps=1,\n        )\n\n        # 2. Define call parameters\n\n        self._guidance_scale = guidance_scale\n        self._clip_skip = 0\n        self._cross_attention_kwargs = None\n        self._denoising_end = denoising_end\n        self._interrupt = False\n\n        # 2. Define call parameters\n        batch_size = 1\n        device = self._execution_device\n\n        # 3. Encode input prompt\n        lora_scale = (\n            self.cross_attention_kwargs.get("scale", None)\n            if self.cross_attention_kwargs is not None\n            else None\n        )\n        prompt_embeds = None\n        negative_prompt_embeds = None\n        pooled_prompt_embeds = None\n        negative_pooled_prompt_embeds = None\n        (\n            prompt_embeds,\n            negative_prompt_embeds,\n            pooled_prompt_embeds,\n            negative_pooled_prompt_embeds,\n        ) = self.encode_prompt(\n            prompt=prompt,\n            prompt_2=prompt_2,\n            device=device,\n            num_images_per_prompt=num_images_per_prompt,\n            do_classifier_free_guidance=self.do_classifier_free_guidance,\n            negative_prompt=negative_prompt,\n            negative_prompt_2=negative_prompt_2,\n            prompt_embeds=prompt_embeds,\n            negative_prompt_embeds=negative_prompt_embeds,\n            pooled_prompt_embeds=pooled_prompt_embeds,\n            negative_pooled_prompt_embeds=negative_pooled_prompt_embeds,\n            lora_scale=lora_scale,\n            clip_skip=self.clip_skip,\n        )\n\n        # 4. Prepare timesteps\n        timesteps, num_inference_steps = retrieve_timesteps(\n            self.scheduler, num_inference_steps, device, timesteps\n        )\n\n        num_channels_latents = self.unet.config.in_channels\n        latents = self.prepare_latents(\n            batch_size * num_images_per_prompt,\n            num_channels_latents,\n            height,\n            width,\n            prompt_embeds.dtype,\n            device,\n            generator,\n            latents,\n        )\n\n        # 6. Prepare extra step kwargs.\n        extra_step_kwargs = self.prepare_extra_step_kwargs(generator, eta)\n\n        # 7. Prepare added time ids & embeddings\n        add_text_embeds = pooled_prompt_embeds\n        if self.text_encoder_2 is None:\n            text_encoder_projection_dim = int(pooled_prompt_embeds.shape[-1])\n        else:\n            text_encoder_projection_dim = self.text_encoder_2.config.projection_dim\n\n        add_time_ids = self._get_add_time_ids(\n            original_size,\n            (0, 0),\n            target_size,\n            dtype=prompt_embeds.dtype,\n            text_encoder_projection_dim=text_encoder_projection_dim,\n        )\n        negative_add_time_ids = add_time_ids\n\n        if self.do_classifier_free_guidance:\n            prompt_embeds = torch.cat([negative_prompt_embeds, prompt_embeds], dim=0)\n            add_text_embeds = torch.cat(\n                [negative_pooled_prompt_embeds, add_text_embeds], dim=0\n            )\n            add_time_ids = torch.cat([negative_add_time_ids, add_time_ids], dim=0)\n\n        prompt_embeds = prompt_embeds.to(device)\n        add_text_embeds = add_text_embeds.to(device)\n        add_time_ids = add_time_ids.to(device).repeat(\n            batch_size * num_images_per_prompt, 1\n        )\n\n        # 8. Denoising loop\n        num_warmup_steps = max(\n            len(timesteps) - num_inference_steps * self.scheduler.order, 0\n        )\n\n        # 8.1 Apply denoising_end\n        if (\n            self.denoising_end is not None\n            and isinstance(self.denoising_end, float)\n            and self.denoising_end > 0\n            and self.denoising_end < 1\n        ):\n            discrete_timestep_cutoff = int(\n                round(\n                    self.scheduler.config.num_train_timesteps\n                    - (self.denoising_end * self.scheduler.config.num_train_timesteps)\n                )\n            )\n            num_inference_steps = len(\n                list(filter(lambda ts: ts >= discrete_timestep_cutoff, timesteps))\n            )\n            timesteps = timesteps[:num_inference_steps]\n\n        # 9. Optionally get Guidance Scale Embedding\n        timestep_cond = None\n        if self.unet.config.time_cond_proj_dim is not None:\n            guidance_scale_tensor = torch.tensor(self.guidance_scale - 1).repeat(\n                batch_size * num_images_per_prompt\n            )\n            timestep_cond = self.get_guidance_scale_embedding(\n                guidance_scale_tensor, embedding_dim=self.unet.config.time_cond_proj_dim\n            ).to(device=device, dtype=latents.dtype)\n\n        self._num_timesteps = len(timesteps)\n        for i, t in enumerate(timesteps):\n            if self._interrupt:\n                continue\n            # expand the latents if we are doing classifier free guidance\n            latent_model_input = (\n                torch.cat([latents] * 2)\n                if self.do_classifier_free_guidance\n                else latents\n            )\n\n            latent_model_input = self.scheduler.scale_model_input(latent_model_input, t)\n\n            # predict the noise residual\n            added_cond_kwargs = {\n                "text_embeds": add_text_embeds,\n                "time_ids": add_time_ids,\n            }\n            noise_pred = self.unet(\n                latent_model_input,\n                t,\n                encoder_hidden_states=prompt_embeds,\n                timestep_cond=timestep_cond,\n                cross_attention_kwargs=self.cross_attention_kwargs,\n                added_cond_kwargs=added_cond_kwargs,\n                return_dict=False,\n            )[0]\n\n            # perform guidance\n            if self.do_classifier_free_guidance:\n                noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)\n                noise_pred = noise_pred_uncond + self.guidance_scale * (\n                    noise_pred_text - noise_pred_uncond\n                )\n                # Delete intermediate tensors to free memory\n                del noise_pred_uncond, noise_pred_text\n\n            # compute the previous noisy sample x_t -> x_t-1\n            latents = self.scheduler.step(\n                noise_pred, t, latents, **extra_step_kwargs, return_dict=False\n            )[0]\n            \n            # Delete intermediate tensors from this iteration\n            del latent_model_input, noise_pred\n\n        if self.is_hyper:\n            latents = latents.to(torch.float32)\n            image = self.decode_latents_tensors(latents)\n            image = image.to(torch.float16)\n        else:\n            image = self.decode_latents_tensors(latents)\n\n        # apply watermark if available\n        if self.watermark is not None:\n            image = self.watermark.apply_watermark(image)\n\n        # Clean up intermediate tensors that are no longer needed\n        del prompt_embeds, add_text_embeds, add_time_ids\n        if timestep_cond is not None:\n            del timestep_cond\n        if negative_prompt_embeds is not None:\n            del negative_prompt_embeds\n        if negative_pooled_prompt_embeds is not None:\n            del negative_pooled_prompt_embeds\n        \n        # Offload all models\n        self.maybe_free_model_hooks()\n        \n        # Clear cache after cleanup\n        torch.cuda.empty_cache()\n\n        return image\n', 'rewards/base_reward.py': 'from abc import ABC, abstractmethod\n\nimport torch\n\n\nclass BaseRewardLoss(ABC):\n    """\n    Base class for reward functions implementing a differentiable reward function for optimization.\n    """\n\n    def __init__(self, name: str, weighting: float):\n        self.name = name\n        self.weighting = weighting\n        self._text_feature_cache = {}  # Cache for text features by prompt\n\n    @staticmethod\n    def freeze_parameters(params: torch.nn.ParameterList):\n        for param in params:\n            param.requires_grad = False\n\n    @abstractmethod\n    def get_image_features(self, image: torch.Tensor) -> torch.Tensor:\n        pass\n\n    @abstractmethod\n    def get_text_features(self, prompt: str) -> torch.Tensor:\n        pass\n\n    @abstractmethod\n    def compute_loss(\n        self, image_features: torch.Tensor, text_features: torch.Tensor\n    ) -> torch.Tensor:\n        pass\n\n    def process_features(self, features: torch.Tensor) -> torch.Tensor:\n        features_normed = features / features.norm(dim=-1, keepdim=True)\n        return features_normed\n    \n    def clear_text_cache(self):\n        """Clear the text feature cache to free memory."""\n        self._text_feature_cache.clear()\n\n    def __call__(self, image: torch.Tensor, prompt: str) -> torch.Tensor:\n        image_features = self.get_image_features(image)\n        \n        # Check cache for text features to avoid recomputing\n        if prompt not in self._text_feature_cache:\n            text_features = self.get_text_features(prompt)\n            self._text_feature_cache[prompt] = text_features\n        else:\n            text_features = self._text_feature_cache[prompt]\n\n        image_features_normed = self.process_features(image_features)\n        text_features_normed = self.process_features(text_features)\n\n        loss = self.compute_loss(image_features_normed, text_features_normed)\n        return loss\n', 'rewards/hps.py': 'import shutil\nfrom pathlib import Path\n\nimport huggingface_hub\nimport torch\n\n\ndef _ensure_hpsv2_clip_vocab() -> None:\n    """The hpsv2==1.2.0 wheel ships its vendored open_clip without the BPE\n    vocab file, so importing it raises FileNotFoundError. The file is\n    byte-identical to the one shipped by openai-clip, so copy it across.\n    Must run before `hpsv2.src.open_clip` is imported, since that subpackage\n    reads the vocab at import time."""\n    import hpsv2\n\n    target = Path(hpsv2.__file__).parent / "src" / "open_clip" / "bpe_simple_vocab_16e6.txt.gz"\n    if target.exists():\n        return\n    import clip\n    source = Path(clip.__file__).parent / "bpe_simple_vocab_16e6.txt.gz"\n    shutil.copyfile(source, target)\n\n\n_ensure_hpsv2_clip_vocab()\n\nfrom hpsv2.src.open_clip import create_model, get_tokenizer\nfrom hpsv2.src.open_clip.model import convert_weights_to_lp\n\nfrom rewards.base_reward import BaseRewardLoss\n\n\nclass HPSLoss(BaseRewardLoss):\n    """HPS reward loss function for optimization."""\n\n    def __init__(\n        self,\n        weighting: float,\n        dtype: torch.dtype,\n        device: torch.device,\n        cache_dir: str,\n    ):\n        precision_str = {\n            torch.bfloat16: "bf16",\n            torch.float16: "fp16",\n            torch.float32: "fp32",\n        }[dtype]\n        self.hps_model = create_model(\n            "ViT-H-14",\n            "laion2B-s32B-b79K",\n            precision=precision_str,\n            device=device,\n            cache_dir=cache_dir,\n        )\n        checkpoint_path = huggingface_hub.hf_hub_download(\n            "xswu/HPSv2", "HPS_v2.1_compressed.pt", cache_dir=cache_dir\n        )\n        self.hps_model.load_state_dict(\n            torch.load(checkpoint_path, map_location=device)["state_dict"]\n        )\n        self.hps_tokenizer = get_tokenizer("ViT-H-14")\n        self.device = device\n        self.hps_model = self.hps_model.to(device)\n        if dtype != torch.float32:\n            convert_weights_to_lp(self.hps_model, dtype=dtype)\n        self.hps_model.eval()\n        self.freeze_parameters(self.hps_model.parameters())\n        super().__init__("HPS", weighting)\n        self.hps_model.set_grad_checkpointing(True)\n\n    def get_image_features(self, image: torch.Tensor) -> torch.Tensor:\n        hps_image_features = self.hps_model.encode_image(image)\n        return hps_image_features\n\n    def get_text_features(self, prompt: str) -> torch.Tensor:\n        hps_text = self.hps_tokenizer(prompt).to(self.device)\n        hps_text_features = self.hps_model.encode_text(hps_text)\n        return hps_text_features\n\n    def compute_loss(\n        self, image_features: torch.Tensor, text_features: torch.Tensor\n    ) -> torch.Tensor:\n        logits_per_image = image_features @ text_features.T\n        hps_loss = 1 - torch.diagonal(logits_per_image).mean()\n        return hps_loss\n\n    def get_per_sample_scores(\n        self, image_features: torch.Tensor, text_features: torch.Tensor\n    ) -> torch.Tensor:\n        """Return per-sample HPS scores (not averaged). Higher = better quality."""\n        logits_per_image = image_features @ text_features.T\n        return torch.diagonal(logits_per_image)\n', 'rewards/utils.py': 'from typing import Any, List\n\nimport torch\nfrom torchvision.transforms import (CenterCrop, Compose, InterpolationMode,\n                                    Normalize, Resize)\nfrom transformers import AutoProcessor\n\nfrom rewards.base_reward import BaseRewardLoss\nfrom rewards.clip import CLIPLoss\nfrom rewards.clip_b32 import CLIPB32Loss\nfrom rewards.hps import HPSLoss\nfrom rewards.imagereward import ImageRewardLoss\nfrom rewards.pickscore import PickScoreLoss\n\n\ndef get_reward_losses(\n    cfg: Any, dtype: torch.dtype, device: torch.device, cache_dir: str\n) -> List[BaseRewardLoss]:\n    # Load tokenizer for CLIP H/14 (used by CLIP and PickScore)\n    if cfg.rewards.clip.enable or cfg.rewards.pickscore.enable:\n        tokenizer = AutoProcessor.from_pretrained(\n            "laion/CLIP-ViT-H-14-laion2B-s32B-b79K", cache_dir=cache_dir\n        )\n\n    # Load tokenizer for CLIP B/32 (used by CLIP_b32)\n    if cfg.rewards.clip_b32.enable:\n        tokenizer_b32 = AutoProcessor.from_pretrained(\n            "openai/clip-vit-base-patch32", cache_dir=cache_dir\n        )\n\n    reward_losses = []\n    if cfg.rewards.hps.enable:\n        reward_losses.append(\n            HPSLoss(cfg.rewards.hps.weighting, dtype, device, cache_dir)\n        )\n    if cfg.rewards.imagereward.enable:\n        reward_losses.append(\n            ImageRewardLoss(\n                cfg.rewards.imagereward.weighting,\n                dtype,\n                device,\n                cache_dir,\n            )\n        )\n    if cfg.rewards.clip.enable:\n        reward_losses.append(\n            CLIPLoss(\n                cfg.rewards.clip.weighting,\n                dtype,\n                device,\n                cache_dir,\n                tokenizer,\n            )\n        )\n    if cfg.rewards.clip_b32.enable:\n        reward_losses.append(\n            CLIPB32Loss(\n                cfg.rewards.clip_b32.weighting,\n                dtype,\n                device,\n                cache_dir,\n                tokenizer_b32,\n            )\n        )\n    if cfg.rewards.pickscore.enable:\n        reward_losses.append(\n            PickScoreLoss(\n                cfg.rewards.pickscore.weighting,\n                dtype,\n                device,\n                cache_dir,\n                tokenizer,\n            )\n        )\n    return reward_losses\n\n\ndef get_all_reward_losses(\n    dtype: torch.dtype, device: torch.device, cache_dir: str\n) -> List[BaseRewardLoss]:\n    """\n    Create instances of ALL available reward losses for comprehensive metric logging.\n    These are used for logging purposes only (computed without gradients).\n\n    Includes: HPS, CLIP H/14, and CLIP B/32\n    (ImageReward and PickScore are excluded to save computation)\n\n    Args:\n        dtype: Data type for models\n        device: Device for computation\n        cache_dir: Cache directory for model weights\n\n    Returns:\n        List of reward loss instances (with weighting=0 since not used for optimization)\n    """\n    # Load tokenizer for CLIP H/14\n    tokenizer = AutoProcessor.from_pretrained(\n        "laion/CLIP-ViT-H-14-laion2B-s32B-b79K", cache_dir=cache_dir\n    )\n\n    # Load tokenizer for CLIP B/32\n    tokenizer_b32 = AutoProcessor.from_pretrained(\n        "openai/clip-vit-base-patch32", cache_dir=cache_dir\n    )\n\n    all_losses = []\n\n    # Create reward losses (weighting=0 since only for logging)\n    all_losses.append(HPSLoss(0.0, dtype, device, cache_dir))\n    all_losses.append(CLIPLoss(0.0, dtype, device, cache_dir, tokenizer))\n    all_losses.append(CLIPB32Loss(0.0, dtype, device, cache_dir, tokenizer_b32))\n\n    return all_losses\n\n\ndef get_all_reward_losses_shared(\n    enabled_losses: List[BaseRewardLoss],\n    dtype: torch.dtype,\n    device: torch.device,\n    cache_dir: str,\n) -> List[BaseRewardLoss]:\n    """\n    Create instances of reward losses for comprehensive metric logging,\n    reusing already-loaded models when possible.\n\n    Args:\n        enabled_losses: List of reward losses already loaded for optimization\n        dtype: Data type for models\n        device: Device for computation\n        cache_dir: Cache directory for model weights\n\n    Returns:\n        List of reward loss instances for comprehensive logging\n    """\n    import logging\n\n    # Get names of already-enabled losses\n    enabled_names = {loss.name for loss in enabled_losses}\n\n    # Required losses for logging: only HPS (CLIP models are too heavy for benchmarks)\n    required = {\'HPS\'}\n    needed = required - enabled_names\n\n    if not needed:\n        logging.info(f"All reward losses already loaded ({len(enabled_losses)}), reusing for logging")\n        return enabled_losses\n\n    # Start with enabled losses (reuse them)\n    all_losses = list(enabled_losses)\n\n    logging.info(f"Reusing {len(enabled_losses)} enabled losses, loading {len(needed)} additional: {needed}")\n\n    # Load tokenizers only if needed\n    tokenizer = None\n    tokenizer_b32 = None\n\n    if \'CLIP\' in needed:\n        tokenizer = AutoProcessor.from_pretrained(\n            "laion/CLIP-ViT-H-14-laion2B-s32B-b79K", cache_dir=cache_dir\n        )\n        all_losses.append(CLIPLoss(0.0, dtype, device, cache_dir, tokenizer))\n\n    if \'CLIP_b32\' in needed:\n        tokenizer_b32 = AutoProcessor.from_pretrained(\n            "openai/clip-vit-base-patch32", cache_dir=cache_dir\n        )\n        all_losses.append(CLIPB32Loss(0.0, dtype, device, cache_dir, tokenizer_b32))\n\n    if \'HPS\' in needed:\n        all_losses.append(HPSLoss(0.0, dtype, device, cache_dir))\n\n    return all_losses\n\n\ndef clip_img_transform(size: int = 224):\n    return Compose(\n        [\n            Resize(size, interpolation=InterpolationMode.BICUBIC),\n            CenterCrop(size),\n            Normalize(\n                (0.48145466, 0.4578275, 0.40821073),\n                (0.26862954, 0.26130258, 0.27577711),\n            ),\n        ]\n    )\n', 'objectives/base_objective.py': 'from abc import ABC, abstractmethod\n\nimport torch\n\n\nclass BaseDiversityObjective(ABC):\n    """\n    Base class for diversity objectives implementing differentiable pairwise diversity functions.\n    """\n\n    def __init__(self, name: str, weighting: float, device: torch.device):\n        self.name = name\n        self.weighting = weighting\n        self.device = device\n\n    @staticmethod\n    def freeze_parameters(params):\n        """Freeze model parameters."""\n        for param in params:\n            param.requires_grad = False\n\n    @abstractmethod\n    def compute_pairwise_diversity(\n        self, images: torch.Tensor\n    ) -> torch.Tensor:\n        """\n        Compute pairwise diversity score for a batch of images.\n        \n        Args:\n            images: Batch of images with shape (B, C, H, W), values in [0, 1]\n            \n        Returns:\n            Average pairwise diversity score (higher = more diverse)\n        """\n        pass\n\n    def __call__(self, images: torch.Tensor) -> torch.Tensor:\n        """\n        Compute diversity loss for a batch of images.\n        \n        Args:\n            images: Batch of images with shape (B, C, H, W), values in [0, 1]\n            \n        Returns:\n            Diversity loss (negative of diversity score for maximization)\n        """\n        if images.shape[0] <= 1:\n            # No diversity to compute for single image\n            return torch.tensor(0.0, device=self.device, dtype=images.dtype)\n        \n        diversity_score = self.compute_pairwise_diversity(images)\n        # Negative sign because we want to maximize diversity (minimize loss)\n        return -diversity_score\n\n', 'objectives/dino_backend.py': '"""Shared DINO backend for multiple diversity objectives.\n\nThis module provides a shared DINO model and preprocessing pipeline\nfor DPP, Vendi, and DINO diversity.\n"""\n\nimport logging\nimport torch\nimport torch.nn.functional as F\nfrom transformers import AutoModel, AutoImageProcessor\n\n\nclass SharedDINOBackend:\n    """Shared DINO model backend for feature extraction.\n    \n    This class loads the DINO model once and provides feature extraction\n    for multiple diversity objectives (DPP, Vendi, DINO).\n    """\n    \n    def __init__(\n        self,\n        model_name: str = "facebook/dinov2-base",\n        device: torch.device = torch.device("cuda"),\n        cache_dir: str = "./model_cache",\n    ):\n        """Initialize the shared DINO backend.\n        \n        Args:\n            model_name: HuggingFace model name for DINO\n            device: Device to load model on\n            cache_dir: Directory to cache downloaded models\n        """\n        self.device = device\n        self.model_name = model_name\n        \n        logging.info(f"[SharedDINOBackend] Loading DINO model: {model_name}")\n        \n        # Load DINO model and processor\n        self.dino_model = AutoModel.from_pretrained(\n            model_name,\n            cache_dir=cache_dir,\n        ).to(device)\n        self.dino_model.eval()\n        \n        self.processor = AutoImageProcessor.from_pretrained(\n            model_name,\n            cache_dir=cache_dir,\n        )\n        \n        # ImageNet normalization constants\n        self._img_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)\n        self._img_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)\n        \n        logging.info(f"[SharedDINOBackend] DINO model loaded successfully on {device}")\n    \n    def preprocess_images(self, images: torch.Tensor) -> torch.Tensor:\n        """Preprocess images for DINO model.\n        \n        Args:\n            images: Tensor of shape (B, C, H, W) in range [0, 1]\n            \n        Returns:\n            Preprocessed images ready for DINO model\n        """\n        # Convert [0, 1] to [-1, 1]\n        images_dino = images * 2.0 - 1.0\n        \n        # Resize to 256x256 as expected by DINO\n        images_processed = F.interpolate(\n            images_dino,\n            size=(256, 256),\n            mode="bilinear",\n            align_corners=False,\n        )\n        \n        # Convert back to [0, 1] for ImageNet normalization\n        images_processed = images_processed * 0.5 + 0.5\n        \n        # Apply ImageNet normalization\n        images_processed = (images_processed - self._img_mean) / self._img_std\n        \n        return images_processed\n    \n    def extract_features(self, images: torch.Tensor, normalize: bool = True, use_cls: bool = True) -> torch.Tensor:\n        """Extract DINO features from images.\n        \n        Args:\n            images: Tensor of shape (B, C, H, W) in range [0, 1]\n            normalize: Whether to L2-normalize the features\n            use_cls: If True, return CLS token (B, D). If False, return patch tokens (B, N, D)\n            \n        Returns:\n            If use_cls=True: CLS token embeddings of shape (B, D)\n            If use_cls=False: Patch token embeddings of shape (B, N, D) where N is number of patches\n        """\n        # Preprocess images\n        images_processed = self.preprocess_images(images)\n        \n        # Extract features\n        with torch.set_grad_enabled(torch.is_grad_enabled()):\n            outputs = self.dino_model(pixel_values=images_processed)\n            features = outputs.last_hidden_state  # (B, num_tokens, D)\n            \n            if use_cls:\n                # Extract CLS token (first token)\n                embeddings = features[:, 0, :]  # (B, D)\n            else:\n                # Extract patch tokens (all except CLS)\n                embeddings = features[:, 1:, :]  # (B, N, D)\n            \n            # Normalize if requested\n            if normalize:\n                embeddings = F.normalize(embeddings, p=2, dim=-1)\n        \n        return embeddings\n    \n    def clear_cache(self):\n        """Clear any cached data and free GPU memory."""\n        torch.cuda.empty_cache()\n    \n    def __repr__(self) -> str:\n        return f"SharedDINOBackend(model={self.model_name}, device={self.device})"\n\n', 'objectives/dpp_diversity.py': 'import torch\nimport torch.nn.functional as F\n\nfrom objectives.base_objective import BaseDiversityObjective\n\n\nclass DPPDiversityObjective(BaseDiversityObjective):\n    """DPP (Determinantal Point Process) diversity objective supporting multiple feature backends."""\n\n    SUPPORTED_BACKENDS = {"dino", "dreamsim"}\n\n    def __init__(\n        self,\n        weighting: float,\n        device: torch.device,\n        dino_backend=None,\n        dreamsim_backend=None,\n        backend_type: str = "dino",\n        epsilon: float = 1e-6,\n    ):\n        """\n        Initialize DPP diversity objective.\n        \n        Args:\n            weighting: Weight for this objective in the total loss\n            device: Device for computation\n            feature_backend: Backend instance that exposes an extract_features(...) method\n            backend_type: Identifier for feature backend (\'dino\' or \'dreamsim\')\n            epsilon: Small jitter added to kernel diagonal for numerical stability\n        """\n        super().__init__("diversity_dpp", weighting, device)\n        self._last_raw_score: float = 0.0  # Store raw score for metrics reporting\n        if backend_type not in self.SUPPORTED_BACKENDS:\n            raise ValueError(\n                f"Unsupported DPP backend \'{backend_type}\'. "\n                f"Supported backends: {sorted(self.SUPPORTED_BACKENDS)}"\n            )\n        if backend_type == "dreamsim":\n            if dreamsim_backend is None:\n                raise ValueError("DreamSim backend requested for DPP but not initialized.")\n            self.feature_backend = dreamsim_backend\n        else:\n            if dino_backend is None:\n                raise ValueError("DINO backend requested for DPP but not initialized.")\n            self.feature_backend = dino_backend\n        self.backend_type = backend_type\n        self.epsilon = epsilon\n\n    def _extract_embeddings(self, images: torch.Tensor) -> torch.Tensor:\n        """Extract normalized embeddings using the configured backend."""\n        if self.backend_type == "dino":\n            embeddings = self.feature_backend.extract_features(\n                images, normalize=True, use_cls=True\n            )\n        elif self.backend_type == "dreamsim":\n            embeddings = self.feature_backend.extract_features(images)\n            if embeddings is None:\n                raise RuntimeError(\n                    "DreamSim backend did not return embeddings for DPP computation."\n                )\n            embeddings = F.normalize(embeddings, p=2, dim=-1)\n        else:\n            raise ValueError(f"Unsupported DPP backend \'{self.backend_type}\'")\n        return embeddings\n\n    def compute_pairwise_diversity(self, images: torch.Tensor) -> torch.Tensor:\n        """\n        Compute normalized DPP diversity score using log-determinant of similarity kernel.\n        \n        Returns normalized score [0, 1]. Raw score stored in _last_raw_score for metrics.\n        """\n        batch_size = images.shape[0]\n        if batch_size <= 1:\n            self._last_raw_score = 0.0\n            return torch.tensor(0.0, device=self.device, dtype=images.dtype)\n\n        cls_embeddings = self._extract_embeddings(images)\n        kernel = cls_embeddings @ cls_embeddings.T\n        kernel = (kernel + kernel.T) * 0.5\n        identity = torch.eye(batch_size, device=self.device, dtype=kernel.dtype)\n        kernel = kernel + self.epsilon * identity\n\n        sign, logdet = torch.linalg.slogdet(identity + kernel)\n        raw_score = logdet if sign.item() > 0 else torch.tensor(0.0, device=self.device, dtype=images.dtype)\n        \n        # Store raw score for metrics, return normalized\n        self._last_raw_score = raw_score.item()\n        max_score = batch_size * torch.log(torch.tensor(2.0, device=self.device, dtype=images.dtype))\n        return raw_score / max_score\n\n', 'training/early_stopping.py': '"""Early stopping utilities for diversity-based training."""\nimport logging\nimport math\nfrom typing import Callable, Dict, Optional, Tuple\n\n\ndef calculate_dynamic_thresholds(\n    rewards: Dict[str, float],\n    diversity_thresholds: Dict[str, float],\n    iteration: int,\n) -> Dict[str, float]:\n    """\n    Calculate dynamic thresholds based on initial diversity values.\n    \n    Args:\n        rewards: Dictionary of current rewards/diversity scores\n        diversity_thresholds: Dictionary containing multiplier settings\n        iteration: Current iteration number (0 = first iteration)\n    \n    Returns:\n        Dictionary mapping metric names to calculated threshold values\n    """\n    if iteration != 0:\n        return {}\n    \n    use_dynamic = diversity_thresholds.get(\'use_dynamic_thresholds\', False)\n    if not use_dynamic:\n        return {}\n    \n    default_multiplier = diversity_thresholds.get(\'dynamic_threshold_multiplier\', 4.0)\n    \n    # Metrics that use dynamic thresholds with their default multipliers\n    dynamic_metrics = {\n        "diversity_tiny_l2": ("tiny_l2_multiplier", 1.5),  # Default 1.5 for Tiny L2\n        "diversity_color": ("color_multiplier", 1.5),  # Default 1.5 for Color\n        "diversity_dpp": ("dpp_multiplier", default_multiplier),\n        "diversity_dpp_patch": ("dpp_patch_multiplier", default_multiplier),\n        "diversity_vendi": ("vendi_multiplier", default_multiplier),\n    }\n    \n    calculated_thresholds = {}\n    initial_values = {}\n    \n    for metric_name, (multiplier_key, metric_default) in dynamic_metrics.items():\n        if metric_name in rewards:\n            initial_value = rewards[metric_name]\n            initial_values[metric_name] = initial_value\n            \n            # Get metric-specific multiplier or use metric-specific default\n            multiplier = diversity_thresholds.get(multiplier_key, metric_default)\n            \n            # Calculate dynamic threshold\n            dynamic_threshold = initial_value * multiplier\n            calculated_thresholds[metric_name] = dynamic_threshold\n            \n            logging.info(\n                f"[Dynamic Threshold] {metric_name}: initial={initial_value:.4f}, "\n                f"multiplier={multiplier:.1f}x, threshold={dynamic_threshold:.4f}"\n            )\n    \n    return calculated_thresholds\n\n\ndef check_early_stopping(\n    rewards: Dict[str, float],\n    diversity_thresholds: Dict[str, float],\n    dynamic_thresholds_calculated: Dict[str, float],\n    initial_diversity_values: Dict[str, float],\n) -> Tuple[bool, Optional[str]]:\n    """\n    Check if early stopping conditions are met.\n    \n    Args:\n        rewards: Dictionary of current rewards/diversity scores\n        diversity_thresholds: Dictionary containing threshold settings\n        dynamic_thresholds_calculated: Pre-calculated dynamic thresholds\n        initial_diversity_values: Initial diversity values from iteration 0\n    \n    Returns:\n        Tuple of (should_stop, stop_reason)\n    """\n    use_dynamic = diversity_thresholds.get(\'use_dynamic_thresholds\', False)\n    \n    # Map diversity objective names to threshold keys and their dynamic behavior\n    # Fixed metrics: LPIPS, DINO, DreamSim - require explicit threshold\n    # Dynamic metrics: TinyL2, Color, DPP, Vendi - automatically use dynamic when enabled\n    threshold_mapping = {\n        "diversity_lpips": ("lpips_threshold", False),         # Fixed only\n        "diversity_dino": ("dino_threshold", False),           # Fixed only\n        "diversity_dreamsim": ("dreamsim_threshold", False),   # Fixed only\n        "diversity_tiny_l2": (None, True),      # Dynamic automatically when enabled\n        "diversity_color": (None, True),       # Dynamic automatically when enabled\n        "diversity_dpp": (None, True),         # Dynamic automatically when enabled\n        "diversity_dpp_patch": (None, True),   # Dynamic automatically when enabled\n        "diversity_vendi": (None, True),       # Dynamic automatically when enabled\n    }\n    \n    for diversity_name, (threshold_key, auto_dynamic) in threshold_mapping.items():\n        if diversity_name not in rewards:\n            continue\n        \n        current_score = rewards[diversity_name]\n        threshold_value = None\n        is_dynamic = False\n        \n        # Dynamic metrics (TinyL2, Color, DPP, Vendi) automatically use dynamic thresholds\n        if auto_dynamic and use_dynamic and diversity_name in dynamic_thresholds_calculated:\n            # Automatically use dynamic threshold\n            threshold_value = dynamic_thresholds_calculated[diversity_name]\n            is_dynamic = True\n        # Fixed metrics (LPIPS, DINO, DreamSim) require explicit threshold\n        elif threshold_key is not None and threshold_key in diversity_thresholds:\n            # Use fixed threshold if specified\n            threshold_value = diversity_thresholds[threshold_key]\n        \n        # Check if threshold is met\n        if threshold_value is not None and current_score >= threshold_value:\n            threshold_type = "dynamic" if is_dynamic else "fixed"\n            stop_reason = (\n                f"{diversity_name} diversity reached {threshold_type} threshold: "\n                f"{current_score:.4f} >= {threshold_value:.4f}"\n            )\n            if is_dynamic:\n                initial_val = initial_diversity_values.get(diversity_name, 0)\n                multiplier = threshold_value / initial_val if initial_val > 0 else 0\n                stop_reason += f" (initial={initial_val:.4f}, {multiplier:.1f}x)"\n            \n            logging.info(f"[Early Stopping] {stop_reason}")\n            return True, stop_reason\n    \n    return False, None\n\n\ndef log_early_stopping(iteration: int, stop_reason: str) -> None:\n    """\n    Log early stopping information.\n    \n    Args:\n        iteration: Iteration number where stopping occurred\n        stop_reason: Reason for stopping\n    """\n    logging.info(f"Stopping optimization at iteration {iteration}: {stop_reason}")\n\n\ndef make_sequential_dpp_stop(\n    num_images: int,\n    threshold: float = 0.03,\n    key: str = "diversity_dpp",\n) -> Callable[[Dict[str, float], int], Optional[str]]:\n    """\n    Build a stop function for sequential DPP sampling that halts when the\n    normalized score under `key` reaches (1 - threshold), i.e. 97% of max by\n    default. Works for both `diversity_dpp` and `diversity_dpp_patch` since\n    both have upper bound num_images * log(2) (BxB cosine kernel logdet).\n    """\n    if num_images < 1:\n        raise ValueError("num_images must be >= 1 for sequential DPP stopping.")\n    max_score = num_images * math.log(2.0)\n    target_normalized = 1.0 - threshold\n\n    def _stop_fn(rewards: Dict[str, float], iteration: int) -> Optional[str]:\n        dpp_value = rewards.get(key)\n        if dpp_value is not None:\n            normalized = dpp_value / max_score\n            if normalized >= target_normalized:\n                return (\n                    f"{key} reached {target_normalized:.0%} of max: "\n                    f"{normalized:.4f} >= {target_normalized:.4f}"\n                )\n        return None\n\n    return _stop_fn\n\n\ndef make_dpp_absolute_stop(threshold: float) -> Callable[[Dict[str, float], int], Optional[str]]:\n    """\n    Build a stop function that halts when raw DPP score reaches a fixed threshold.\n    """\n    def _stop_fn(rewards: Dict[str, float], iteration: int) -> Optional[str]:\n        dpp_value = rewards.get("diversity_dpp")\n        if dpp_value is not None and dpp_value >= threshold:\n            return f"diversity_dpp reached absolute threshold: {dpp_value:.4f} >= {threshold:.4f}"\n        return None\n\n    return _stop_fn\n\n\ndef make_dpp_multiplier_stop(multiplier: float) -> Callable[[Dict[str, float], int], Optional[str]]:\n    """\n    Build a stop function that halts when DPP reaches multiplier * initial_dpp.\n\n    Self-resets on iteration 0 so the same closure can be safely reused\n    across multiple `trainer.train()` calls; otherwise `initial_dpp` from a\n    previous run would silently determine the target for the next.\n    """\n    initial_dpp = None\n\n    def _stop_fn(rewards: Dict[str, float], iteration: int) -> Optional[str]:\n        nonlocal initial_dpp\n        dpp_value = rewards.get("diversity_dpp")\n        if dpp_value is None:\n            return None\n\n        # Re-anchor on iteration 0 so reusing this closure across runs is safe.\n        if iteration == 0 or initial_dpp is None:\n            initial_dpp = dpp_value\n            logging.info(f"[DPP Multiplier Stop] Initial DPP: {initial_dpp:.4f}, target: {initial_dpp * multiplier:.4f}")\n            return None\n\n        target = initial_dpp * multiplier\n        if dpp_value >= target:\n            return f"diversity_dpp reached {multiplier}x initial: {dpp_value:.4f} >= {target:.4f}"\n        return None\n\n    return _stop_fn\n\n', 'training/hps_quality_guard.py': 'import logging\nfrom typing import Dict, List, Optional, Tuple\n\nimport torch\n\nfrom rewards.base_reward import BaseRewardLoss\n\n\nclass HPSQualityGuard:\n    """Per-sample HPS quality guard for latent optimization.\n\n    Tracks per-sample HPS scores against per-sample thresholds and either\n    freezes (monotonically) or reverts samples whose quality has degraded.\n    Owns its state across iterations of a single train() call.\n\n    `restored_mask` returned by `apply()` flags samples whose latent buffer\n    is no longer the one the optimizer accumulated grads/momentum against —\n    the caller must zero those samples\' grad and optimizer state before the\n    next step.\n    """\n\n    def __init__(\n        self,\n        latents: torch.Tensor,\n        device: torch.device,\n        reward_losses: List[BaseRewardLoss],\n        sample_threshold: Optional[float],\n        absolute_threshold: Optional[float],\n        revert_mode: bool,\n    ):\n        self.device = device\n        self.sample_threshold = sample_threshold\n        self.absolute_threshold = absolute_threshold\n        self.revert_mode = revert_mode\n\n        self.num_samples = latents.shape[0]\n        self.frozen_samples = torch.zeros(self.num_samples, dtype=torch.bool, device=device)\n        self.last_good_latents = latents.detach().clone()\n        self.revert_counts = torch.zeros(self.num_samples, dtype=torch.int32, device=device)\n        self.per_sample_hps_thresholds: Optional[torch.Tensor] = None\n\n        self.hps_reward_loss: Optional[BaseRewardLoss] = None\n        if sample_threshold is not None:\n            for rl in reward_losses:\n                if rl.name == "HPS":\n                    self.hps_reward_loss = rl\n                    break\n            if self.hps_reward_loss is None:\n                logging.warning("hps_sample_threshold specified but HPS reward not enabled. Ignoring.")\n\n    @property\n    def enabled(self) -> bool:\n        """True iff the guard should run this train() call."""\n        return self.sample_threshold is not None and self.hps_reward_loss is not None\n\n    def apply(\n        self,\n        preprocessed_image: torch.Tensor,\n        prompt: str,\n        latents: torch.Tensor,\n    ) -> Tuple[bool, Optional[str], Dict[str, float], List[str], Optional[torch.Tensor]]:\n        """Run the guard for one iteration.\n\n        Computes per-sample HPS scores, lazily initializes per-sample\n        thresholds on the first call, and dispatches to the freeze or revert\n        branch. Mutates internal state and `latents.data` in place.\n\n        Returns (should_stop, stop_reason, rewards_extras, log_fragments,\n        restored_mask). `should_stop` is only ever True from the freeze path\n        (when every sample is frozen).\n        """\n        with torch.no_grad():\n            image_features = self.hps_reward_loss.get_image_features(preprocessed_image)\n            text_features = self.hps_reward_loss.get_text_features(prompt)\n            per_sample_scores = self.hps_reward_loss.get_per_sample_scores(image_features, text_features)\n\n            if self.per_sample_hps_thresholds is None:\n                initial_scores = per_sample_scores.detach().clone()\n                if self.absolute_threshold is not None:\n                    self.per_sample_hps_thresholds = torch.full_like(initial_scores, self.absolute_threshold)\n                    logging.info(f"[HPS Freeze] Using absolute threshold: {self.absolute_threshold}")\n                else:\n                    self.per_sample_hps_thresholds = torch.minimum(\n                        torch.full_like(initial_scores, self.sample_threshold),\n                        initial_scores - 0.01,\n                    )\n                logging.info(f"[HPS Freeze] Initial scores: {initial_scores.tolist()}")\n                logging.info(f"[HPS Freeze] Per-sample thresholds (stop when <=): {self.per_sample_hps_thresholds.tolist()}")\n\n            below_threshold = per_sample_scores <= self.per_sample_hps_thresholds\n\n            if self.revert_mode:\n                rewards_extras, log_fragments, restored_mask = self._apply_revert(\n                    below_threshold, per_sample_scores, latents\n                )\n                return False, None, rewards_extras, log_fragments, restored_mask\n            return self._apply_freeze(below_threshold, per_sample_scores, latents)\n\n    def _apply_revert(\n        self,\n        below_threshold: torch.Tensor,\n        per_sample_scores: torch.Tensor,\n        latents: torch.Tensor,\n    ) -> Tuple[Dict[str, float], List[str], Optional[torch.Tensor]]:\n        """For each sample below threshold: restore latents.data[idx] to\n        last_good_latents[idx] and increment revert_counts[idx]. For samples\n        above threshold: refresh last_good_latents. Never stops.\n\n        The returned restored_mask is the per-iteration `below_threshold`\n        set whose grad+optimizer state must be zeroed before the next step\n        (else Adam/SGD momentum from the divergent step keeps moving the\n        just-restored latent in the bad direction).\n        """\n        if below_threshold.any():\n            revert_indices = torch.where(below_threshold)[0].tolist()\n            for idx in revert_indices:\n                logging.info(\n                    f"[HPS Revert] Sample {idx} quality degraded below threshold "\n                    f"{self.per_sample_hps_thresholds[idx].item():.4f} "\n                    f"(score: {per_sample_scores[idx].item():.4f})"\n                )\n                latents.data[idx] = self.last_good_latents[idx].clone()\n                self.revert_counts[idx] += 1\n                logging.info(\n                    f"[HPS Revert] Reverted sample {idx} to last good latents "\n                    f"(revert #{self.revert_counts[idx].item()})"\n                )\n\n        good_samples = ~below_threshold\n        if good_samples.any():\n            for idx in torch.where(good_samples)[0]:\n                self.last_good_latents[idx] = latents.data[idx].detach().clone()\n\n        total_reverts = self.revert_counts.sum().item()\n        restored_mask = below_threshold if below_threshold.any() else None\n        return ({"total_reverts": total_reverts}, [f"[Reverts: {total_reverts}]"], restored_mask)\n\n    def _apply_freeze(\n        self,\n        below_threshold: torch.Tensor,\n        per_sample_scores: torch.Tensor,\n        latents: torch.Tensor,\n    ) -> Tuple[bool, Optional[str], Dict[str, float], List[str], Optional[torch.Tensor]]:\n        """Newly-frozen samples (below threshold this iter, not previously\n        frozen) get their latents restored to the last known good state.\n        The frozen mask is monotonically growing. Triggers should_stop when\n        every sample is frozen.\n\n        The returned restored_mask is the cumulative `frozen_samples` set:\n        those latents must keep their grad+optimizer state zeroed every\n        iteration so they truly stop moving (otherwise momentum keeps\n        accumulating from non-zero gradients on the live forward pass).\n        """\n        just_frozen = below_threshold & ~self.frozen_samples\n        if just_frozen.any():\n            frozen_indices = torch.where(just_frozen)[0].tolist()\n            for idx in frozen_indices:\n                logging.info(\n                    f"[HPS Freeze] Sample {idx} quality degraded below threshold "\n                    f"{self.per_sample_hps_thresholds[idx].item():.4f} "\n                    f"(score: {per_sample_scores[idx].item():.4f})"\n                )\n                latents.data[idx] = self.last_good_latents[idx].clone()\n                logging.info(f"[HPS Freeze] Restored sample {idx} to last good latents")\n        self.frozen_samples = self.frozen_samples | below_threshold\n\n        good_samples = ~self.frozen_samples\n        if good_samples.any():\n            for idx in torch.where(good_samples)[0]:\n                self.last_good_latents[idx] = latents.data[idx].detach().clone()\n\n        num_frozen = self.frozen_samples.sum().item()\n        rewards_extras = {"frozen_samples": num_frozen}\n        log_fragments = [f"[Frozen: {num_frozen}/{self.num_samples}]"]\n\n        should_stop = bool(self.frozen_samples.all())\n        stop_reason = None\n        if should_stop:\n            stop_reason = f"All {self.num_samples} samples degraded below HPS thresholds"\n            logging.info(f"[HPS Freeze] {stop_reason}")\n\n        restored_mask = self.frozen_samples if self.frozen_samples.any() else None\n        return should_stop, stop_reason, rewards_extras, log_fragments, restored_mask\n', 'training/trainer.py': 'import logging\nimport math\nfrom typing import Callable, Dict, List, Optional, Set, Tuple\n\nimport PIL\nimport PIL.Image\nimport torch\nfrom diffusers import DiffusionPipeline\n\nfrom rewards import clip_img_transform\nfrom rewards.base_reward import BaseRewardLoss\nfrom objectives.base_objective import BaseDiversityObjective\nfrom training.early_stopping import (\n    calculate_dynamic_thresholds,\n    check_early_stopping,\n    log_early_stopping,\n)\nfrom training.hps_quality_guard import HPSQualityGuard\n\n\nclass LatentNoiseTrainer:\n    """Trainer for optimizing latents with reward losses."""\n\n    def __init__(\n        self,\n        reward_losses: List[BaseRewardLoss],\n        model: DiffusionPipeline,\n        n_iters: int,\n        n_inference_steps: int,\n        seed: int,\n        regularize: bool = True,\n        regularization_weight: float = 0.01,\n        grad_clip: float = 0.1,\n        log_metrics: bool = True,\n        device: torch.device = torch.device("cuda"),\n        diversity_objectives: Optional[List[BaseDiversityObjective]] = None,\n        diversity_thresholds: Optional[Dict[str, float]] = None,\n        all_reward_losses: Optional[List[BaseRewardLoss]] = None,\n        all_diversity_objectives: Optional[List[BaseDiversityObjective]] = None,\n    ):\n        self.reward_losses = reward_losses\n        self.diversity_objectives = diversity_objectives if diversity_objectives is not None else []\n        # Comprehensive metrics for logging (all available metrics, not just enabled ones)\n        self.all_reward_losses = all_reward_losses if all_reward_losses is not None else reward_losses\n        self.all_diversity_objectives = all_diversity_objectives if all_diversity_objectives is not None else self.diversity_objectives\n        self.model = model\n        self.n_iters = n_iters\n        self.n_inference_steps = n_inference_steps\n        self.seed = seed\n        self.regularize = regularize\n        self.regularization_weight = regularization_weight\n        self.grad_clip = grad_clip\n        self.log_metrics = log_metrics\n        self.device = device\n        self.preprocess_fn = clip_img_transform(224)\n        self.diversity_thresholds = diversity_thresholds if diversity_thresholds is not None else {}\n        self._base_diversity_objectives = list(self.diversity_objectives)\n        self.diversity_reference_images: Optional[torch.Tensor] = None\n        self.diversity_reference_objectives: Optional[Set[str]] = None\n\n    @staticmethod\n    def _tensor_to_pil(batch: torch.Tensor) -> List[PIL.Image.Image]:\n        image_numpy = batch.detach().cpu().permute(0, 2, 3, 1).float().numpy()\n        return DiffusionPipeline.numpy_to_pil(image_numpy)\n\n    def _compute_all_metrics(self, image: torch.Tensor, prompt: str, load_all_metrics: bool = True) -> Dict[str, float]:\n        """Compute all reward and diversity metrics for a given image.\n\n        This computes ALL available metrics (not just enabled ones) without gradients\n        for comprehensive logging purposes.\n\n        Args:\n            image: Image tensor to compute metrics for\n            prompt: Text prompt associated with the image\n            load_all_metrics: If True, temporarily load all metrics for comprehensive logging\n\n        Returns:\n            Dictionary of all computed metrics\n        """\n        metrics = {}\n\n        # Compute without gradients for logging\n        with torch.no_grad():\n            preprocessed_image = self.preprocess_fn(image)\n\n            # Compute ALL reward losses (including disabled ones)\n            for reward_loss in self.all_reward_losses:\n                try:\n                    loss = reward_loss(preprocessed_image, prompt)\n                    if reward_loss.name == "HPS":\n                        metrics[reward_loss.name] = 1 - loss.item()\n                    elif reward_loss.name == "CLIP":\n                        metrics[reward_loss.name] = 1 - loss.item() / 100\n                    else:\n                        metrics[reward_loss.name] = loss.item()\n                except Exception as e:\n                    logging.warning(f"Failed to compute {reward_loss.name}: {e}")\n                    metrics[reward_loss.name] = float(\'nan\')\n\n            # Compute ALL diversity objectives (including disabled ones)\n            for diversity_obj in self.all_diversity_objectives:\n                try:\n                    diversity_loss = diversity_obj(image)\n                    if hasattr(diversity_obj, \'_last_raw_score\'):\n                        metrics[diversity_obj.name] = diversity_obj._last_raw_score\n                    else:\n                        metrics[diversity_obj.name] = -diversity_loss.item()\n                except Exception as e:\n                    logging.warning(f"Failed to compute {diversity_obj.name}: {e}")\n                    metrics[diversity_obj.name] = float(\'nan\')\n\n            # If we loaded extra metrics, compute them and then clean up\n            if load_all_metrics and hasattr(self, \'_temp_logging_objectives\'):\n                for diversity_obj in self._temp_logging_objectives:\n                    try:\n                        diversity_loss = diversity_obj(image)\n                        if hasattr(diversity_obj, \'_last_raw_score\'):\n                            metrics[diversity_obj.name] = diversity_obj._last_raw_score\n                        else:\n                            metrics[diversity_obj.name] = -diversity_loss.item()\n                    except Exception as e:\n                        logging.warning(f"Failed to compute {diversity_obj.name}: {e}")\n                        metrics[diversity_obj.name] = float(\'nan\')\n\n            # Delete preprocessed_image to free memory\n            del preprocessed_image\n\n        return metrics\n\n    # ------------------------------------------------------------------ #\n    # train() helpers — pure-as-possible extractions of train() phases.  #\n    # ------------------------------------------------------------------ #\n\n    def _generate_image(self, current_latents: torch.Tensor, prompt: str) -> torch.Tensor:\n        """Run the diffusion pipeline forward with a fresh seeded generator.\n\n        Calls torch.cuda.empty_cache() internally to release intermediate\n        activations, matching the original train() behavior.\n        """\n        generator = torch.Generator("cuda").manual_seed(self.seed)\n        image = self.model.apply(\n            latents=current_latents,\n            prompt=prompt,\n            generator=generator,\n            num_inference_steps=self.n_inference_steps,\n            num_images_per_prompt=current_latents.shape[0],\n        )\n        del generator\n        torch.cuda.empty_cache()\n        return image\n\n    def _capture_initial(\n        self,\n        image: torch.Tensor,\n        current_latents: torch.Tensor,\n        prompt: str,\n        multi_apply_fn,\n    ) -> Tuple[List[PIL.Image.Image], Dict[str, float]]:\n        """Compute the iteration-0 \'initial\' images and their full metric set.\n\n        If multi_apply_fn is provided it is used to render the initial batch\n        from the latents (multi-step rendering); otherwise the just-generated\n        single-step image is reused.\n        """\n        if multi_apply_fn is not None:\n            initial_batch = multi_apply_fn(current_latents.detach(), prompt)\n        else:\n            initial_batch = image.detach()\n        initial_images = self._tensor_to_pil(initial_batch)\n        initial_rewards = self._compute_all_metrics(initial_batch, prompt)\n        logging.info(f"Initial metrics: {initial_rewards}")\n        del initial_batch\n        torch.cuda.empty_cache()\n        return initial_images, initial_rewards\n\n    def _compute_reward_losses(\n        self,\n        image: torch.Tensor,\n        prompt: str,\n        hps_in_warmup: bool,\n        total_loss: torch.Tensor,\n    ) -> Tuple[torch.Tensor, Dict[str, float], List[str], torch.Tensor]:\n        """Run all reward losses, accumulate into total_loss, build log fragments.\n\n        HPS during warmup is logged with `(warmup)` suffix on the same fragment\n        and is NOT added to total_loss.\n\n        Returns the updated total_loss, a per-reward score dict, log fragments,\n        and the preprocessed image tensor (so the HPS quality guard can reuse\n        it without preprocessing twice).\n        """\n        rewards: Dict[str, float] = {}\n        log_fragments: List[str] = []\n        preprocessed_image = self.preprocess_fn(image)\n        for reward_loss in self.reward_losses:\n            loss = reward_loss(preprocessed_image, prompt).to(total_loss.dtype)\n            if reward_loss.name == "HPS":\n                score = 1 - loss.item()\n            elif reward_loss.name == "CLIP":\n                score = 1 - loss.item() / 100\n            else:\n                score = loss.item()\n            frag = f"{reward_loss.name}: {score:.4f}"\n            if reward_loss.name == "HPS" and hps_in_warmup:\n                frag += "(warmup)"\n            else:\n                total_loss = total_loss + loss * reward_loss.weighting\n            log_fragments.append(frag)\n            rewards[reward_loss.name] = score\n        return total_loss, rewards, log_fragments, preprocessed_image\n\n    def _compute_diversity_losses(\n        self,\n        image: torch.Tensor,\n        total_loss: torch.Tensor,\n    ) -> Tuple[torch.Tensor, Dict[str, float], List[str]]:\n        """Run all diversity objectives, accumulate into total_loss.\n\n        Honors `self.diversity_reference_images` and the per-objective\n        membership in `self.diversity_reference_objectives` so that sequential\n        rounds prepend reference images to the diversity input.\n        """\n        rewards: Dict[str, float] = {}\n        log_fragments: List[str] = []\n        for diversity_obj in self.diversity_objectives:\n            diversity_input = image\n            use_references = (\n                self.diversity_reference_images is not None\n                and (\n                    self.diversity_reference_objectives is None\n                    or diversity_obj.name in self.diversity_reference_objectives\n                )\n            )\n            if use_references:\n                reference_batch = self.diversity_reference_images.to(\n                    image.device, dtype=image.dtype\n                )\n                reference_batch = reference_batch.detach()\n                diversity_input = torch.cat([reference_batch, image], dim=0)\n            diversity_loss = diversity_obj(diversity_input).to(total_loss.dtype)\n            if hasattr(diversity_obj, \'_last_raw_score\'):\n                rewards[diversity_obj.name] = diversity_obj._last_raw_score\n            else:\n                rewards[diversity_obj.name] = -diversity_loss.item()\n            log_fragments.append(f"{diversity_obj.name}: {-diversity_loss.item():.4f}")\n            total_loss = total_loss + diversity_obj.weighting * diversity_loss\n        return total_loss, rewards, log_fragments\n\n    def _init_dynamic_thresholds_if_first(\n        self,\n        iteration: int,\n        rewards: Dict[str, float],\n        dynamic_thresholds_calculated: Dict,\n        initial_diversity_values: Dict[str, float],\n    ) -> None:\n        """On iteration 0, compute dynamic thresholds and snapshot initial\n        diversity values. Mutates the two accumulator dicts in place.\n        """\n        if iteration != 0:\n            return\n        calculated = calculate_dynamic_thresholds(\n            rewards, self.diversity_thresholds, iteration\n        )\n        dynamic_thresholds_calculated.update(calculated)\n        for metric_name in calculated.keys():\n            if metric_name in rewards:\n                initial_diversity_values[metric_name] = rewards[metric_name]\n\n    def _check_stops(\n        self,\n        rewards: Dict[str, float],\n        iteration: int,\n        dynamic_thresholds_calculated: Dict,\n        initial_diversity_values: Dict[str, float],\n        extra_stop_fn: Optional[Callable[[Dict[str, float], int], Optional[str]]],\n        should_stop: bool,\n        stop_reason: Optional[str],\n    ) -> Tuple[bool, Optional[str]]:\n        """Combine built-in early-stopping checks with the optional\n        extra_stop_fn callback. Preserves the original semantics: if HPS\n        already requested a stop, that wins; built-in early stops fire next;\n        the custom callback can override only if neither has fired.\n        """\n        if not should_stop:\n            should_stop, stop_reason = check_early_stopping(\n                rewards,\n                self.diversity_thresholds,\n                dynamic_thresholds_calculated,\n                initial_diversity_values,\n            )\n        if not should_stop and extra_stop_fn is not None:\n            try:\n                custom_stop_reason = extra_stop_fn(rewards, iteration)\n            except Exception as extra_exc:\n                logging.warning(f"Custom stop function raised an error: {extra_exc}")\n                custom_stop_reason = None\n            if custom_stop_reason:\n                should_stop = True\n                stop_reason = custom_stop_reason\n                logging.info(f"[Custom Stop] {custom_stop_reason}")\n        return should_stop, stop_reason\n\n    def _apply_regularization(\n        self,\n        current_latents: torch.Tensor,\n        total_loss: torch.Tensor,\n        rewards: Dict[str, float],\n        latent_dim: int,\n    ) -> torch.Tensor:\n        """Add the spherical-boundary regularization term to total_loss.\n\n        Computed in fp32 to avoid overflow. Mutates `rewards` to add the\n        "norm" entry (mean per-sample latent L2 norm).\n        """\n        flat_latents = current_latents.view(current_latents.shape[0], -1)\n        latent_norm = torch.linalg.vector_norm(flat_latents, dim=1).to(torch.float32)\n        safe_norm = torch.clamp(latent_norm, min=1e-6)\n        regularization_per_sample = (\n            0.5 * safe_norm**2 - (latent_dim - 1) * torch.log(safe_norm)\n        )\n        regularization = self.regularization_weight * regularization_per_sample.mean()\n        rewards["norm"] = latent_norm.mean().item()\n        total_loss = total_loss + regularization.to(total_loss.dtype)\n        del flat_latents, latent_norm, safe_norm, regularization_per_sample, regularization\n        return total_loss\n\n    def _record_iteration(\n        self,\n        iteration: int,\n        rewards: Dict[str, float],\n        total_loss: torch.Tensor,\n        return_history: bool,\n        iteration_history: Optional[List[Dict[str, float]]],\n    ) -> None:\n        """Append per-iteration metrics to history. No-op if not recording."""\n        if not return_history:\n            return\n        iter_metrics = {\n            "iteration": iteration,\n            "loss/total": total_loss.item(),\n        }\n        iter_metrics.update({f"reward/{key}": value for key, value in rewards.items()})\n        iteration_history.append(iter_metrics)\n\n    @staticmethod\n    def _update_best(\n        image: torch.Tensor,\n        latents: torch.Tensor,\n        rewards: Dict[str, float],\n        total_reward_loss: float,\n        best_loss: float,\n        best_image: Optional[torch.Tensor],\n        best_rewards: Optional[Dict[str, float]],\n        best_latents: Optional[torch.Tensor],\n    ) -> Tuple[float, Optional[torch.Tensor], Optional[Dict[str, float]], Optional[torch.Tensor], bool]:\n        """If the current iteration improved on best_loss, snapshot the image,\n        rewards, and latents. Must be called BEFORE backward()."""\n        is_best_image = total_reward_loss < best_loss\n        if is_best_image:\n            best_loss = total_reward_loss\n            best_image = image.detach().clone() if hasattr(image, \'detach\') else image\n            best_rewards = rewards\n            best_latents = latents.detach().cpu()\n        return best_loss, best_image, best_rewards, best_latents, is_best_image\n\n    @staticmethod\n    def _zero_optimizer_state(\n        latents: torch.Tensor,\n        mask: torch.Tensor,\n        optimizer: torch.optim.Optimizer,\n    ) -> None:\n        """Zero gradients and optimizer momentum/variance buffers along the\n        leading (sample) dimension of `latents` for the indices flagged in\n        `mask`.\n\n        Required for freeze and revert modes: when `latents.data[idx]` has\n        been overwritten from a snapshot, the gradient just produced by\n        backward() and any Adam/SGD momentum carrying state from the\n        divergent step would otherwise immediately move the restored latent\n        in the wrong direction on the next optimizer.step().\n\n        Covers SGD-with-momentum (`momentum_buffer`) and Adam variants\n        (`exp_avg`, `exp_avg_sq`). Other state keys are left alone.\n        """\n        if mask is None or not mask.any():\n            return\n        idxs = torch.where(mask)[0]\n        if latents.grad is not None:\n            for idx in idxs:\n                latents.grad.data[idx] = 0.0\n        for param_group in optimizer.param_groups:\n            for p in param_group[\'params\']:\n                if p is latents and p in optimizer.state:\n                    state = optimizer.state[p]\n                    for key in (\'momentum_buffer\', \'exp_avg\', \'exp_avg_sq\'):\n                        buf = state.get(key)\n                        if buf is not None:\n                            for idx in idxs:\n                                buf[idx] = 0.0\n\n    def _step_optimizer(\n        self,\n        total_loss: torch.Tensor,\n        latents: torch.Tensor,\n        optimizer: torch.optim.Optimizer,\n        image: torch.Tensor,\n        is_best_image: bool,\n        iteration: int,\n        restored_mask: Optional[torch.Tensor] = None,\n    ) -> None:\n        """Run backward + (optional) per-reward CPU offload + grad clip +\n        zero-out for restored samples + optimizer step. Cleans up `image`\n        and `total_loss` and empties the cache.\n\n        `restored_mask` flags samples whose latent buffer was overwritten by\n        the HPS quality guard this iteration; their gradient and optimizer\n        momentum/variance are zeroed before stepping so the restoration\n        actually sticks.\n        """\n        total_loss.backward()\n\n        # After backward pass, offload reward models to CPU if using lazy loading\n        for reward_loss in self.reward_losses:\n            if hasattr(reward_loss, \'offload_after_backward\'):\n                reward_loss.offload_after_backward()\n\n        torch.nn.utils.clip_grad_norm_(latents, self.grad_clip)\n        self._zero_optimizer_state(latents, restored_mask, optimizer)\n        optimizer.step()\n\n        # Clear memory after optimizer step - delete image and other large tensors.\n        # Note: best_image was already cloned above; safe to delete the working copy.\n        if iteration != self.n_iters - 1 and not is_best_image:\n            del image\n        del total_loss\n        torch.cuda.empty_cache()\n\n    def _finalize_best(\n        self,\n        best_image: torch.Tensor,\n        best_latents: Optional[torch.Tensor],\n        best_rewards: Optional[Dict[str, float]],\n        prompt: str,\n        multi_apply_fn,\n        skip_final_metrics_log: bool,\n    ) -> Tuple[List[PIL.Image.Image], Dict[str, float]]:\n        """Convert the best tensor to PIL and recompute its metrics. If a\n        multi-step renderer is provided, re-render the best latents with it\n        and use that image for the final metrics.\n        """\n        best_image_pil = self._tensor_to_pil(best_image)\n        if multi_apply_fn is not None and best_latents is not None:\n            multi_step_image = multi_apply_fn(best_latents.to("cuda"), prompt)\n            best_image_pil = self._tensor_to_pil(multi_step_image)\n            best_rewards = self._compute_all_metrics(multi_step_image, prompt)\n            if not skip_final_metrics_log:\n                logging.info(f"Best metrics (after multi-step): {best_rewards}")\n            del multi_step_image\n        else:\n            best_rewards = self._compute_all_metrics(best_image, prompt)\n            if not skip_final_metrics_log:\n                logging.info(f"Best metrics: {best_rewards}")\n        return best_image_pil, best_rewards\n\n    # ------------------------------------------------------------------ #\n    # Public training loop. Reads top-to-bottom as a sequence of phases. #\n    # ------------------------------------------------------------------ #\n\n    def train(\n        self,\n        latents: torch.Tensor,\n        prompt: str,\n        optimizer: torch.optim.Optimizer,\n        save_dir: Optional[str] = None,\n        multi_apply_fn=None,\n        return_history: bool = False,\n        extra_stop_fn: Optional[Callable[[Dict[str, float], int], Optional[str]]] = None,\n        skip_final_metrics_log: bool = False,\n        hps_sample_threshold: Optional[float] = None,\n        hps_absolute_threshold: Optional[float] = None,\n        hps_revert_mode: bool = False,\n        hps_warmup_iters: int = 0,\n    ) -> Tuple[List[PIL.Image.Image], List[PIL.Image.Image], Dict[str, float], Dict[str, float], Optional[List[Dict[str, float]]]]:\n        logging.info(f"Optimizing latents for prompt \'{prompt}\'.")\n        best_loss = torch.inf\n        best_image = best_latents = best_rewards = None\n        initial_images = initial_rewards = None\n        iteration_history = [] if return_history else None\n        initial_diversity_values: Dict[str, float] = {}\n        dynamic_thresholds_calculated: Dict = {}\n        use_dynamic = self.diversity_thresholds.get(\'use_dynamic_thresholds\', False)\n        default_multiplier = self.diversity_thresholds.get(\'dynamic_threshold_multiplier\', 4.0)\n        latent_dim = math.prod(latents.shape[1:])\n\n        hps_guard = HPSQualityGuard(\n            latents=latents,\n            device=self.device,\n            reward_losses=self.reward_losses,\n            sample_threshold=hps_sample_threshold,\n            absolute_threshold=hps_absolute_threshold,\n            revert_mode=hps_revert_mode,\n        )\n\n        for iteration in range(self.n_iters):\n            log_fragments: List[str] = []\n            rewards: Dict[str, float] = {}\n            should_stop = False\n            stop_reason: Optional[str] = None\n            restored_mask: Optional[torch.Tensor] = None\n            optimizer.zero_grad()\n\n            current_latents = latents\n            image = self._generate_image(current_latents, prompt)\n\n            if initial_images is None:\n                initial_images, initial_rewards = self._capture_initial(\n                    image, current_latents, prompt, multi_apply_fn\n                )\n\n            total_loss = torch.zeros((), device=image.device, dtype=latents.dtype, requires_grad=False)\n            hps_in_warmup = iteration < hps_warmup_iters\n            total_loss, r, frags, preprocessed_image = self._compute_reward_losses(\n                image, prompt, hps_in_warmup, total_loss\n            )\n            rewards.update(r)\n            log_fragments.extend(frags)\n\n            if hps_guard.enabled:\n                should_stop, stop_reason, r, frags, restored_mask = hps_guard.apply(\n                    preprocessed_image, prompt, latents,\n                )\n                rewards.update(r)\n                log_fragments.extend(frags)\n\n            total_loss, r, frags = self._compute_diversity_losses(image, total_loss)\n            rewards.update(r)\n            log_fragments.extend(frags)\n\n            self._init_dynamic_thresholds_if_first(\n                iteration, rewards, dynamic_thresholds_calculated, initial_diversity_values\n            )\n            should_stop, stop_reason = self._check_stops(\n                rewards, iteration, dynamic_thresholds_calculated,\n                initial_diversity_values, extra_stop_fn, should_stop, stop_reason,\n            )\n\n            rewards["total"] = total_loss.item()\n            total_reward_loss = total_loss.item()\n            if self.regularize:\n                total_loss = self._apply_regularization(current_latents, total_loss, rewards, latent_dim)\n\n            if self.log_metrics:\n                logging.info(f"Iteration {iteration}: {\', \'.join(log_fragments)}")\n            self._record_iteration(iteration, rewards, total_loss, return_history, iteration_history)\n\n            best_loss, best_image, best_rewards, best_latents, is_best_image = self._update_best(\n                image, latents, rewards, total_reward_loss,\n                best_loss, best_image, best_rewards, best_latents,\n            )\n\n            if should_stop:\n                log_early_stopping(iteration, stop_reason)\n                if best_image is None:\n                    best_image = image.detach().clone() if hasattr(image, \'detach\') else image\n                    best_rewards = rewards\n                    best_latents = latents.detach().cpu()\n                break\n\n            if iteration != self.n_iters - 1:\n                self._step_optimizer(\n                    total_loss, latents, optimizer, image, is_best_image, iteration,\n                    restored_mask=restored_mask,\n                )\n\n        best_image_pil, best_rewards = self._finalize_best(\n            best_image, best_latents, best_rewards, prompt, multi_apply_fn, skip_final_metrics_log,\n        )\n        del best_image, best_latents\n        if \'image\' in locals():\n            del image\n        if \'latents\' in locals():\n            del latents\n        torch.cuda.empty_cache()\n\n        return initial_images, best_image_pil, initial_rewards, best_rewards, iteration_history\n', 'training/noise_utils.py': '"""Utility functions for different noise initialization strategies."""\n\nimport logging\n\nimport torch\nimport numpy as np\n\n\ndef _generate_2d_pink_noise(batch_size, channels, height, width, device, dtype, generator, exponent):\n    """\n    Generate 2D pink noise via spectral filtering: 1/(1+f)^exponent.\n\n    Matches the paper formulation:\n        z_white ~ N(0, I)\n        z_hat   = FFT2D(z_white)\n        z_hat_pink(u, v) = z_hat(u, v) / (1 + sqrt(u^2 + v^2))^alpha\n        z_pink  = (IFFT2D(z_hat_pink) - mu) / sigma     (per-channel)\n    The (1+f) term keeps DC at unit gain, so no explicit DC zeroing is needed.\n    """\n    pink_noise = torch.randn(batch_size, channels, height, width, device=\'cpu\',\n                             dtype=torch.float32, generator=generator)\n\n    if exponent == 0.0:\n        # White noise: still apply per-channel mean/std normalization for consistency\n        flat = pink_noise.reshape(batch_size * channels, height * width)\n        flat = (flat - flat.mean(dim=1, keepdim=True)) / (flat.std(dim=1, keepdim=True) + 1e-8)\n        return flat.reshape(batch_size, channels, height, width).to(device=device, dtype=dtype)\n\n    # Radial frequency grid in integer index units (matches u, v in the paper).\n    freq_h = torch.fft.fftfreq(height).view(height, 1) * height\n    freq_w = torch.fft.rfftfreq(width).view(1, width // 2 + 1) * width\n    freq_radial = torch.sqrt(freq_h**2 + freq_w**2)\n\n    scaling = 1.0 / (1.0 + freq_radial) ** exponent\n\n    for b in range(batch_size):\n        for c in range(channels):\n            noise_fft = torch.fft.rfft2(pink_noise[b, c])\n            pink_noise[b, c] = torch.fft.irfft2(noise_fft * scaling, s=(height, width))\n\n    pink_noise = pink_noise.reshape(batch_size * channels, height * width)\n    pink_noise = (pink_noise - pink_noise.mean(dim=1, keepdim=True)) / (pink_noise.std(dim=1, keepdim=True) + 1e-8)\n    pink_noise = pink_noise.reshape(batch_size, channels, height, width)\n\n    return pink_noise.to(device=device, dtype=dtype)\n\n\ndef _seq_len_to_2d(seq_len):\n    """Convert a flattened sequence length to 2D spatial dimensions."""\n    sqrt_seq = int(seq_len ** 0.5)\n    if sqrt_seq * sqrt_seq == seq_len:\n        return sqrt_seq, sqrt_seq\n    for h in [16, 32, 64, 128]:\n        if seq_len % h == 0:\n            w = seq_len // h\n            if w <= 256:\n                return h, w\n    return sqrt_seq, seq_len // sqrt_seq\n\n\ndef generate_pink_noise(shape, device, dtype=torch.float32, seed=None, exponent=1.0):\n    """\n    Generate pink noise using 1/(1+f)^exponent spectral filtering on a 2D grid.\n\n    Per-sample seeding (seed + i) matches the white-noise convention so the only\n    difference between white and pink for the same seed is the spectral filter.\n\n    Args:\n        shape: (B, seq_len, C) for sequence-shaped latents (3D path), or\n               (B, C, H, W) for image-shaped latents (4D path).\n        device: Device to create tensor on.\n        dtype: Data type of the tensor.\n        seed: Random seed for reproducibility (optional).\n        exponent: Amplitude spectrum exponent (0=white, 1=pink). alpha in [0, 1].\n\n    Returns:\n        Tensor of pink noise with the specified shape.\n    """\n    if len(shape) == 3:\n        # Sequence-shaped latents: route through a 2D grid so spatial correlations\n        # survive when the consumer interprets the sequence as packed image tokens.\n        batch_size, seq_len, channels = shape\n        height, width = _seq_len_to_2d(seq_len)\n\n        samples = []\n        for i in range(batch_size):\n            gen = torch.Generator(device=\'cpu\')\n            if seed is not None:\n                gen.manual_seed(seed + i)\n            pink_2d = _generate_2d_pink_noise(1, channels, height, width,\n                                              device, dtype, gen, exponent)\n            pink_seq = pink_2d.permute(0, 2, 3, 1).reshape(1, seq_len, channels).contiguous()\n            samples.append(pink_seq)\n        return torch.cat(samples, dim=0)\n\n    elif len(shape) == 4:\n        batch_size, channels, height, width = shape\n\n        samples = []\n        for i in range(batch_size):\n            gen = torch.Generator(device=\'cpu\')\n            if seed is not None:\n                gen.manual_seed(seed + i)\n            samples.append(_generate_2d_pink_noise(1, channels, height, width,\n                                                    device, dtype, gen, exponent))\n        return torch.cat(samples, dim=0)\n\n    else:\n        raise ValueError(f"Unsupported shape: {shape}. Expected 3D or 4D tensor.")\n\n\ndef generate_latents(shape, device, dtype, noise_type=\'white\', seed=None, noise_exponent=1.0,\n                     flux_schnell_pack=False):\n    """\n    Generate initial latents with specified noise type.\n\n    Args:\n        shape: Shape of the latent tensor. For Flux-schnell with `flux_schnell_pack=True`,\n            pass the 4D pre-pack shape (B, 16, 64, 64) at 512x512; the result is packed\n            to (B, 1024, 64). For Flux-klein, pass the 4D pre-pack shape and leave\n            `flux_schnell_pack=False` (the klein pipeline packs internally), so the 2D\n            spatial pink correlations are preserved end-to-end.\n        device: Device to create tensor on.\n        dtype: Data type of the tensor.\n        noise_type: \'white\' or \'pink\'.\n        seed: Random seed for reproducibility (optional).\n        noise_exponent: Amplitude spectrum exponent for pink noise (alpha in [0, 1]).\n        flux_schnell_pack: If True, generate noise in the given 4D shape, then apply\n            Flux\'s 2x2 spatial->channel packing so spatial pink correlations survive\n            into the packed sequence consumed by Flux-schnell.\n\n    Returns:\n        Tensor of latents with specified noise type.\n    """\n    if noise_type == \'white\':\n        if seed is not None:\n            # Per-sample CPU generator (seed + i) so white and pink share the seeding\n            # convention.\n            batch_size = shape[0]\n            single_shape = (1,) + tuple(shape[1:])\n            samples = []\n            for i in range(batch_size):\n                gen = torch.Generator(device=\'cpu\').manual_seed(seed + i)\n                samples.append(torch.randn(single_shape, device=\'cpu\', dtype=dtype, generator=gen))\n            latents = torch.cat(samples, dim=0).to(device)\n        else:\n            latents = torch.randn(shape, device=device, dtype=dtype)\n    elif noise_type == \'pink\':\n        latents = generate_pink_noise(shape, device, dtype, seed=seed, exponent=noise_exponent)\n    else:\n        raise ValueError(f"Unknown noise type: {noise_type}. Choose \'white\' or \'pink\'.")\n\n    if flux_schnell_pack:\n        if latents.dim() != 4:\n            raise ValueError(\n                f"flux_schnell_pack=True requires a 4D shape (B, C, H, W); got {tuple(latents.shape)}"\n            )\n        b, c, h, w = latents.shape\n        # 2x2 spatial -> channel packing, matching diffusers FluxPipeline._pack_latents.\n        latents = latents.view(b, c, h // 2, 2, w // 2, 2)\n        latents = latents.permute(0, 2, 4, 1, 3, 5)\n        latents = latents.reshape(b, (h // 2) * (w // 2), c * 4)\n\n    if noise_type == \'pink\':\n        l32 = latents.detach().to(torch.float32)\n        m, s = l32.mean().item(), l32.std().item()\n        if not (np.isfinite(m) and np.isfinite(s)) or abs(m) > 0.05 or not (0.9 < s < 1.1):\n            logging.warning(\n                f"[generate_latents] pink stats off: mean={m:.4f} std={s:.4f} "\n                f"(expected ~0 / ~1; exponent={noise_exponent})"\n            )\n\n    return latents\n', 'training/optim.py': 'import torch\n\n\ndef get_optimizer(optimizer_name: str, latents: torch.Tensor, lr: float):\n    if optimizer_name == "adam":\n        optimizer = torch.optim.Adam([latents], lr=lr, eps=1e-2)\n    elif optimizer_name == "sgd":\n        optimizer = torch.optim.SGD([latents], lr=lr, momentum=0.9)\n    elif optimizer_name == "lbfgs":\n        optimizer = torch.optim.LBFGS(\n            [latents],\n            lr=lr,\n            max_iter=10,\n            history_size=3,\n            line_search_fn="strong_wolfe",\n        )\n    else:\n        raise ValueError(f"Unknown optimizer {optimizer_name}")\n    return optimizer\n', 'configs/geneval_sdxl_pink.yaml': '# Paper Tab. 2 - SDXL-Turbo, GenEval, pink noise (alpha=0.2), DPP + HPSv2\n# Hyperparameters: lambda_div=150 (DPP), lambda_q=3 (HPS), lr=6, gc=0.1, n_iters=150\n\npaths:\n  cache_dir: "./cache"\n  save_dir: "./outputs"\n\nmodel:\n  name: "sdxl-turbo"\n  dtype: "float16"\n  device_id: 0\n  cpu_offloading: false\n\noptimization:\n  lr: 6.0\n  n_iters: 150\n  n_inference_steps: 1\n  num_samples: 4\n  optim: "sgd"\n  grad_clip: 0.1\n  seed: 42\n  noise_type: "pink"\n  noise_exponent: 0.2\n\nregularization:\n  enable: true\n  weight: 0.01\n\nrewards:\n  hps:\n    enable: true\n    weighting: 3.0\n  clip:\n    enable: false\n\ndiversity:\n  dino:\n    enable: false\n  dpp:\n    enable: true\n    weight: 150.0\n    backend: "dino"\n\ntask:\n  type: "geneval"\n  prompt_start_index: null\n  prompt_end_index: null\n\nmulti_step:\n  enable: false\n', 'environment-divgen.yml': 'name: divgen\nchannels:\n  - pytorch\n  - nvidia\n  - conda-forge\ndependencies:\n  - python=3.11\n  - pip\n  - pytorch=2.5.1\n  - pytorch-cuda=12.1\n  - pip:\n      # setuptools 81+ removed pkg_resources, which pytorch-lightning 2.2.5 imports.\n      - "setuptools<81"\n      - pytorch-lightning==2.2.5\n      - torchvision==0.20.1\n      # diffusers>=0.37 needed for Flux2KleinPipeline; pulls transformers>=4.46.\n      - diffusers==0.37.1\n      - transformers==4.56.2\n      - tokenizers==0.22.2\n      - huggingface-hub==0.36.2\n      - accelerate==1.10.1\n      - numpy==2.0.1\n      - pillow==11.0.0\n      - tqdm==4.67.3\n      - pyyaml==6.0.3\n      - blobfile==3.1.0\n      - hpsv2==1.2.0\n      - image-reward==1.5\n      # Required transitively by image-reward (its vendored AestheticScore imports `clip`).\n      - openai-clip==1.0.1\n      - xformers==0.0.29.post1\n      - lpips==0.1.4\n      - dreamsim==0.2.1\n', 'models/model_utils.py': 'import logging\nimport os\nfrom typing import Any\nimport torch\nfrom diffusers import (\n    AutoencoderKL,\n    DDPMScheduler,\n    EulerAncestralDiscreteScheduler,\n    Transformer2DModel,\n)\n\nfrom models.RewardPixart import RewardPixartPipeline\nfrom models.RewardStableDiffusionXL import RewardStableDiffusionXL\nfrom models.RewardFlux import RewardFluxPipeline\nfrom models.RewardFlux2Klein import RewardFlux2KleinPipeline\nfrom models.utils import freeze_params\n\n\ndef get_model(\n    model_name: str,\n    dtype: torch.dtype,\n    device: torch.device,\n    cache_dir: str,\n    enable_sequential_cpu_offload: bool = False,\n):\n    logging.info(f"Loading model: {model_name}")\n    if model_name == "sdxl-turbo":\n        vae = AutoencoderKL.from_pretrained(\n            "madebyollin/sdxl-vae-fp16-fix",\n            torch_dtype=torch.float16,\n            cache_dir=cache_dir,\n        )\n        pipe = RewardStableDiffusionXL.from_pretrained(\n            "stabilityai/sdxl-turbo",\n            vae=vae,\n            torch_dtype=dtype,\n            variant="fp16",\n            use_safetensors=True,\n            cache_dir=cache_dir,\n        )\n        pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(\n            pipe.scheduler.config, timestep_spacing="trailing"\n        )\n        pipe = pipe.to(device, dtype)\n    elif model_name == "pixart":\n        pipe = RewardPixartPipeline.from_pretrained(\n            "PixArt-alpha/PixArt-XL-2-1024-MS",\n            torch_dtype=dtype,\n            cache_dir=cache_dir,\n        )\n        pipe.transformer = Transformer2DModel.from_pretrained(\n            "PixArt-alpha/PixArt-Alpha-DMD-XL-2-512x512",\n            subfolder="transformer",\n            torch_dtype=dtype,\n            cache_dir=cache_dir,\n        )\n        pipe.scheduler = DDPMScheduler.from_pretrained(\n            "PixArt-alpha/PixArt-Alpha-DMD-XL-2-512x512",\n            subfolder="scheduler",\n            cache_dir=cache_dir,\n        )\n\n        # speed-up T5 (optional optimization)\n        #pipe.text_encoder.to_bettertransformer()\n        \n        pipe.transformer.eval()\n        freeze_params(pipe.transformer.parameters())\n        pipe.transformer.enable_gradient_checkpointing()\n        pipe = pipe.to(device)\n    elif model_name == "flux-schnell":\n        # Get HuggingFace token from environment variable\n        hf_token = os.environ.get("HF_TOKEN", None)\n        pipe = RewardFluxPipeline.from_pretrained(\n            "black-forest-labs/FLUX.1-schnell",\n            torch_dtype=torch.bfloat16,\n            cache_dir=cache_dir,\n            token=hf_token,\n        )\n        pipe.to(device, torch.bfloat16)\n    elif model_name == "flux-klein":\n        hf_token = os.environ.get("HF_TOKEN", None)\n        pipe = RewardFlux2KleinPipeline.from_pretrained(\n            "black-forest-labs/FLUX.2-klein-4B",\n            torch_dtype=torch.bfloat16,\n            cache_dir=cache_dir,\n            token=hf_token,\n        )\n        pipe.to(device, torch.bfloat16)\n    else:\n        raise ValueError(f"Unknown model name: {model_name}")\n    if enable_sequential_cpu_offload:\n        pipe.enable_sequential_cpu_offload()\n    return pipe\n\n\ndef get_multi_apply_fn(seed: int, pipe: Any):\n    """Multi-step rendering helper for flux schnell. Renders the initial and\n    best images at 4 inference steps even when optimization runs at 1 step."""\n    generator = torch.Generator("cuda").manual_seed(seed)\n\n    @torch.no_grad()\n    def _multi_apply(latents, prompt):\n        return pipe.apply(\n            latents=latents,\n            prompt=prompt,\n            num_inference_steps=4,\n            generator=generator,\n            num_images_per_prompt=latents.shape[0],\n        )\n\n    return _multi_apply\n'}
OFFICIAL_SOURCE_HASHES = {'LICENSE': '42032dfba9f6faadd84c6210f430a67ced365aaec12ec56d75a25acababd06ca', 'models/utils.py': '9e7cd3681ed0b4b91b3c25dd43bfaaecb5458bc99f3be3f89283668d63e7b599', 'models/RewardStableDiffusionXL.py': '5bcb1c34d75ca6a69ed462174d6bd8a300ffb7c9d54336331f8a38b54507569c', 'rewards/base_reward.py': '07b911782b32b6ab042a2ed086446ab46b5bd9e2b38e411c2cefee5ed8fa9cb5', 'rewards/hps.py': 'bbf87c83114129567caa92d3e71a7bb84c9c4f057a3f55a28c05337a0814e4a1', 'rewards/utils.py': 'bd46d71d09de95f37f4fd0148ec0c607729acbffc1fa1780c0b3b267ba639727', 'objectives/base_objective.py': '72e431d91c81bc38e05eb58f603320fb93a12ca568ee5e9ad80bba9b32b34553', 'objectives/dino_backend.py': '730248fe1abc2ded49db0d95b9d279520e6cea3e060ec6658ac31a92ad41f218', 'objectives/dpp_diversity.py': '887bf7bcc50d68a5682b2a6ea1b1fb1f66c7ced123bdccc65ee650f325079329', 'training/early_stopping.py': 'd5fbe6cee233b2d6de02b1770dfd66ae273aab377843af2ef9ba12ecd4c8f190', 'training/hps_quality_guard.py': '462beeb4b7a1354390408ee51a6e45c10ae39e9a68b226fc9a78b3dfa30114a3', 'training/trainer.py': '6033e68e7d6e1efa1d4825ed6dcca8f996fe0c9501d02c0368c24dc9c5147a83', 'training/noise_utils.py': '024d75880a75423417af0928f9284c7b60c1f2419cf12ea594980aca1443e530', 'training/optim.py': '804343d8df023d6b0a6fb25c1ea47aa3ddb002276eb6d181f2a89fed3a53b167', 'configs/geneval_sdxl_pink.yaml': 'b805ae75b2afc783004c59736f419a2457cb41a38fcde80b21410159abb31de1', 'environment-divgen.yml': '9f3e7b38be269d500362d8b959a8a31e64bee52cba90a32a6b5e2e3f6c622ce3', 'models/model_utils.py': '72f95e41f17fc73bf1333cbb47094360a189e2a9081a771bb4cd0be53074fd48'}


In [ ]:
SHARED_SOURCE_HASHES = {'generation_config.py': '32becfa7ff658fbc5d94b7a700c50e39353c289cf52e50bc97f116492b96b6bf', 'floor_sampling.py': 'a67e3a655fae0df3b3e46d87cf3b08dfcdf6159fc3ae2279cc1e76a01c54b81f', 'generation_adapter.py': '73d252f7863685f15b50e00a933b32667c5170f4a0021f8818c3ec9c8d579929', 'initialization.py': '97e12e6c38c105f965980fb529be7ea60ae9ede15fda5c0da7a0415767871e04', 'observer.py': 'cb3b84ae853a13ed888e0d0b45f59d4906df1609bfb62d34deb8fc39eeb6d73f', 'run_helpers.py': 'fa3a11e36f7cd7c470a9c3116894d5d89aef7268a486c754081db12a787232ca', 'run.py': 'df9edaa0b0399283b36845b0bacb83cfc3883450a9c51c1c4cc49d2c9f5e8d48'}
import ast
import types
from torchvision.transforms import Compose, Resize, CenterCrop, Normalize, InterpolationMode

def load_official_sdxl_reference():
    """Execute verified upstream definitions; omit imports for unused model/reward branches."""
    module_name = '_floorrefine_divgen_sdxl_reference'
    ref = types.ModuleType(module_name)
    sys.modules[module_name] = ref
    ref.__dict__.update(Compose=Compose, Resize=Resize, CenterCrop=CenterCrop,
                        Normalize=Normalize, InterpolationMode=InterpolationMode)
    source_root = PROJECT_ROOT/'denoising_optimization/_vendor/divgen'/DIVGEN_REVISION
    local_roots = {'models', 'rewards', 'objectives', 'training'}
    selections = [
        ('models/utils.py', None),
        ('models/RewardStableDiffusionXL.py', None),
        ('models/model_utils.py', None),
        ('rewards/base_reward.py', None),
        ('rewards/hps.py', None),
        ('rewards/utils.py', {'clip_img_transform'}),
        ('objectives/base_objective.py', None),
        ('objectives/dino_backend.py', None),
        ('objectives/dpp_diversity.py', None),
        ('training/early_stopping.py', None),
        ('training/hps_quality_guard.py', None),
        ('training/noise_utils.py', None),
        ('training/optim.py', None),
        ('training/trainer.py', None),
    ]
    for relative, source in OFFICIAL_SOURCES.items():
        if hashlib.sha256(source.encode('utf-8')).hexdigest() != OFFICIAL_SOURCE_HASHES[relative]:
            raise RuntimeError(f'Embedded upstream source changed: {relative}')
        path = source_root/relative
        path.parent.mkdir(parents=True, exist_ok=True)
        if path.exists() and path.read_text(encoding='utf-8') != source:
            raise RuntimeError(f'Local upstream source differs from the notebook: {path}')
        if not path.exists(): path.write_text(source, encoding='utf-8')
    for relative, names in selections:
        tree = ast.parse(OFFICIAL_SOURCES[relative], filename=str(source_root/relative))
        if names is not None:
            tree.body = [n for n in tree.body if isinstance(n, (ast.FunctionDef, ast.ClassDef)) and n.name in names]
        else:
            # Dependencies from the selected path already exist in this isolated namespace.
            # Every function and class body is left unchanged.
            tree.body = [n for n in tree.body if not (
                isinstance(n, ast.ImportFrom) and (n.module or '').split('.')[0] in local_roots)]
        exec(compile(tree, str(source_root/relative), 'exec'), ref.__dict__)
    return ref

DIVGEN = load_official_sdxl_reference()
print('Official SDXL implementation loaded. Source files:', len(OFFICIAL_SOURCES))

from diffusers import AutoencoderKL, StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler
from torch.utils.checkpoint import checkpoint

def freeze(module):
    if module is not None:
        module.eval().requires_grad_(False)
    return module

def encode_condition(prompt):
    # Text encoders leave GPU before UNet optimization. No Accelerate offload hooks
    # run during backward/checkpoint recomputation.
    for encoder in [pipe.text_encoder, pipe.text_encoder_2]:
        if encoder is not None:
            encoder.to(DEVICE)
    with torch.no_grad():
        embeds, _, pooled, _ = pipe.encode_prompt(prompt=prompt, device=DEVICE,
            num_images_per_prompt=1, do_classifier_free_guidance=False)
        tids = pipe._get_add_time_ids((HEIGHT, WIDTH), (0, 0), (HEIGHT, WIDTH),
            dtype=embeds.dtype, text_encoder_projection_dim=pooled.shape[-1]).to(DEVICE)
    for encoder in [pipe.text_encoder, pipe.text_encoder_2]:
        if encoder is not None:
            encoder.to('cpu')
    gc.collect(); torch.cuda.empty_cache()
    pipe.unet.to(DEVICE, dtype=MODEL_DTYPE)
    return {'encoder_hidden_states': embeds.detach().to(dtype=MODEL_DTYPE),
            'added_cond_kwargs': {'text_embeds': pooled.detach().to(dtype=MODEL_DTYPE),
                                 'time_ids': tids.detach().to(dtype=MODEL_DTYPE)}}

def new_scheduler(start=0):
    sched = EulerAncestralDiscreteScheduler.from_config(SCHED_CONFIG)
    sched.set_timesteps(NUM_STEPS, device=DEVICE)
    sched.set_begin_index(start)
    return sched

def resolve_project_root():
    import sys
    cwd = Path.cwd().resolve()
    candidates = [Path(PROJECT_ROOT).expanduser().resolve()] if PROJECT_ROOT else [cwd, *cwd.parents]
    project = next((p for p in candidates if (p/'noise_init/noise_methods.py').is_file()
                    and (p/'pc_specific_psd/_compat_common.py').is_file()), None)
    if project is None:
        raise FileNotFoundError('Set PROJECT_ROOT to the project root containing noise_init and pc_specific_psd.')
    if str(project) not in sys.path:
        sys.path.insert(0, str(project))
    from pc_specific_psd._compat_common import load_noise_init_module
    module = load_noise_init_module('noise_methods')
    if Path(module.__file__).resolve() != (project/'noise_init/noise_methods.py').resolve():
        raise RuntimeError('A different project copy was imported; restart the kernel and set PROJECT_ROOT.')
    return project, module

def initial_noise(seeds, init_name, base_white=None):
    _, module = resolve_project_root()
    h, w = HEIGHT // 8, WIDTH // 8
    if base_white is None:
        base_white = torch.cat([torch.randn(1, 4, h, w, dtype=torch.float32,
            generator=torch.Generator('cpu').manual_seed(s)) for s in seeds])
    if base_white.shape != (4, 4, h, w):
        raise ValueError('Base Gaussian shape does not match the four-image group.')
    base = base_white.detach().float().cpu().clone()
    spec = INIT_CONFIGS[init_name]
    if spec['method'] == 'white':
        raw = base
    elif spec['method'] == 'pink':
        raw = module.pink(base, spec['alpha'])
    elif spec['method'] == 'same_phase_whitefloor':
        raw = module.same_phase_floor(base, spec['alpha'], spec['gamma'])
    else:
        raise ValueError(f'Unknown initialization: {spec}')
    return module.normalize(raw, NORMALIZATION_PROFILE).contiguous()

def sample_segment(state, start, end, condition, *, rng_state=None, seed=None,
                   with_grad=False, capture=False):
    """Official scheduler math, fresh scheduler/RNG per rollout. No graph detaches in grad path."""
    scheduler = new_scheduler(start)
    generator = torch.Generator(device='cpu')
    if rng_state is not None:
        generator.set_state(rng_state.clone())
    else:
        generator.manual_seed(int(seed))
    x = state.to(device=DEVICE, dtype=MODEL_DTYPE)
    predictions, states = [], []
    if capture:
        states.append(x.detach().float().cpu())
    with torch.set_grad_enabled(with_grad):
        for index in range(start, end):
            timestep = scheduler.timesteps[index]
            model_input = scheduler.scale_model_input(x, timestep)
            # Bind this timestep in the closure: backward may recompute after loop end.
            def predict(inp, t=timestep):
                return pipe.unet(inp, t, **condition, return_dict=False)[0]
            if with_grad and GRAD_CHECKPOINT:
                eps = checkpoint(predict, model_input, use_reentrant=False,
                                 preserve_rng_state=True)
            else:
                eps = predict(model_input)
            out = scheduler.step(eps, timestep, x, generator=generator, return_dict=True)
            x = out.prev_sample
            if not torch.isfinite(x).all():
                raise RuntimeError(f'Non-finite values at denoising step {index + 1}.')
            if capture:
                states.append(x.detach().float().cpu())
                predictions.append(out.pred_original_sample.detach().float().cpu())
    return {'final': x.float(), 'rng': generator.get_state().clone(),
            'states': states, 'predictions': predictions}

class SharedHPSLoss(DIVGEN.HPSLoss):
    """Cast preprocessed FP32 pixels to the official HPS image encoder's dtype."""
    def get_image_features(self, image):
        return super().get_image_features(image.to(self.hps_model.visual.conv1.weight.dtype))

class SharedGenerationAdapter:
    """FloorRefine sampling with a connected graph for the official DivGen trainer."""
    def prepare(self, item):
        self.prompt = item['prompt']
        self.seeds = [int(item['batch_seed'])+i+ROLLOUT_SEED_OFFSET for i in range(NUM_IMAGES)]
        self.condition = encode_condition(self.prompt)

    def apply(self, latents, prompt, generator=None, num_inference_steps=None,
              num_images_per_prompt=None):
        if prompt != self.prompt or num_inference_steps != NUM_STEPS or len(latents) != NUM_IMAGES:
            raise ValueError('The prepared prompt and shared generation settings must match.')
        images, finals = [], []
        with_grad = torch.is_grad_enabled() and latents.requires_grad
        for i, seed in enumerate(self.seeds):
            # Match FloorRefine's cast, scheduler scaling and per-image random stream.
            state = latents[i:i+1].to(DEVICE, MODEL_DTYPE)*new_scheduler().init_noise_sigma
            final = sample_segment(state, 0, NUM_STEPS, self.condition,
                                   seed=seed, with_grad=with_grad)['final']
            def decode(value):
                raw = pipe.vae.decode(value.float()/pipe.vae.config.scaling_factor,
                                      return_dict=False)[0]
                return (raw/2+0.5).clamp(0, 1)
            image = (checkpoint(decode, final, use_reentrant=False, preserve_rng_state=True)
                     if with_grad and GRAD_CHECKPOINT else decode(final))
            if not torch.isfinite(image).all():
                raise RuntimeError('Non-finite FP32 decoded image.')
            images.append(image)
            finals.append(final.detach())
        self.last_final_latents = torch.cat(finals).cpu()
        # The DPP objective still sees all four images together. No staged VJP is used.
        return torch.cat(images)

def load_project_noise_module():
    import sys
    if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
    from pc_specific_psd._compat_common import load_noise_init_module
    module = load_noise_init_module('noise_methods')
    expected = (PROJECT_ROOT/'noise_init/noise_methods.py').resolve()
    if Path(module.__file__).resolve() != expected:
        raise RuntimeError('Noise functions came from another project copy. Restart the kernel and set PROJECT_ROOT.')
    return module

def shared_base_white(item):
    return torch.cat([torch.randn((1, 4, HEIGHT//8, WIDTH//8), dtype=torch.float32,
        generator=torch.Generator('cpu').manual_seed(int(item['batch_seed'])+i)) for i in range(NUM_IMAGES)])

def make_initial_noise(item, init_name, base_white):
    seeds = [int(item['batch_seed'])+i for i in range(NUM_IMAGES)]
    fp32 = initial_noise(seeds, init_name, base_white)
    return fp32, fp32.to(device=DEVICE, dtype=DTYPE).detach()

def load_official_models():
    global pipe, hps_reward, dino_backend, dpp_objective, generation_adapter, SCHED_CONFIG
    if DEVICE != 'cuda' or not torch.cuda.is_available():
        raise RuntimeError('Generation requires a CUDA GPU.')
    release_models()
    start = sync_time()
    vae = AutoencoderKL.from_pretrained(VAE_ID, torch_dtype=torch.float32,
        cache_dir=CACHE_DIR, low_cpu_mem_usage=True)
    pipe = StableDiffusionXLPipeline.from_pretrained(MODEL_ID, vae=vae,
        torch_dtype=MODEL_DTYPE, cache_dir=CACHE_DIR, add_watermarker=False, low_cpu_mem_usage=True)
    pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config, timestep_spacing='trailing')
    for model in [pipe.unet, pipe.vae, pipe.text_encoder, pipe.text_encoder_2]:
        freeze(model)
    pipe.unet.to(DEVICE, dtype=MODEL_DTYPE)
    pipe.vae.to(DEVICE, dtype=torch.float32)
    pipe.vae.enable_slicing()
    SCHED_CONFIG = dict(pipe.scheduler.config)
    if SCHED_CONFIG['prediction_type']!='epsilon':
        raise RuntimeError('The shared protocol requires epsilon prediction.')
    generation_adapter = SharedGenerationAdapter()
    device = torch.device(DEVICE)
    hps_reward = SharedHPSLoss(HPS_WEIGHT, DTYPE, device, CACHE_DIR)
    dino_backend = DIVGEN.SharedDINOBackend(DINO_ID, device, CACHE_DIR)
    dpp_objective = DIVGEN.DPPDiversityObjective(DPP_WEIGHT, device, dino_backend=dino_backend)
    return dict(model_loading_seconds=sync_time()-start, memory=memory_snapshot(),
                upstream_revision=DIVGEN_REVISION, execution='official trainer with shared generation',
                scheduler_config=SCHED_CONFIG,
                parameter_dtypes={name:sorted({str(p.dtype) for p in model.parameters()})
                    for name,model in [('unet',pipe.unet),('vae',pipe.vae),
                                       ('hps',hps_reward.hps_model),('dino',dino_backend.dino_model)]})

def tensor_digest(value):
    return hashlib.sha256(value.detach().contiguous().cpu().numpy().tobytes()).hexdigest()

def initial_precision_report(base, fp32, converted):
    restored = converted.detach().float().cpu()
    original = fp32.detach().float().cpu()
    errors = (restored-original).flatten(1)
    relative = errors.norm(dim=1)/original.flatten(1).norm(dim=1).clamp_min(1e-12)
    if not torch.isfinite(restored).all() or float(relative.max()) > 1e-3:
        raise RuntimeError('Initial-noise conversion is non-finite or exceeds the FP16 rounding bound.')
    return dict(base_white_sha256=tensor_digest(base), initial_fp32_sha256=tensor_digest(original),
        initial_optimization_dtype=str(converted.dtype), initial_converted_sha256=tensor_digest(converted),
        conversion_max_abs_error=float(errors.abs().max()), conversion_relative_l2=relative.tolist(),
        fp32_mean_per_channel=original.flatten(2).mean(2).tolist(),
        fp32_std_per_channel=original.flatten(2).std(2, unbiased=False).tolist())


## 4. Check the initialisations

These CPU checks verify that both initialisations use the same Gaussian samples and the project's noise functions. Generation runs independently and saves its own unoptimised baselines.


In [ ]:
item=GENEVAL50[0]
base=shared_base_white(item)
assert torch.equal(base,shared_base_white(item))
module=load_project_noise_module()
seeds=[item['batch_seed']+i for i in range(NUM_IMAGES)]
for name,spec in INIT_CONFIGS.items():
    raw=(module.pink(base.clone(),spec['alpha']) if name=='pink' else
         module.same_phase_floor(base.clone(),spec['alpha'],spec['gamma']))
    expected=module.normalize(raw,NORMALIZATION_PROFILE).contiguous()
    actual=initial_noise(seeds,name,base)
    assert torch.equal(actual,expected) and torch.isfinite(actual).all()
print('CPU checks passed: shared Gaussian samples and project noise construction.')


## 5. Define optimisation and recording

The recorder inherits DivGen's official `train()` method, objective evaluation, best-candidate selection and optimiser step. Only generation is routed through the shared protocol. All four image graphs remain connected to the initial noise until the official `backward()` call.

Each group records loss, gradient norms, actual noise changes and timing. Prompt encoding is included in group setup. Optimisation time covers generation, reward evaluation and backward updates; observer exports are reported separately. Full warm end-to-end time includes group setup and observation, while model loading is recorded separately.

Completed groups are validated and reused. Interrupted groups restart from their initial noise. The run identity separates this protocol from other configurations.


In [ ]:
class RecordedDivGenTrainer(DIVGEN.LatentNoiseTrainer):
    """Record the official trainer under the shared generation protocol."""
    def __init__(self, *args, on_baseline=None, on_budget=None, on_iteration=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.on_baseline, self.on_budget, self.on_iteration = on_baseline, on_budget, on_iteration
        self.iteration = 0
        self.best_iteration = 0
        self.optimization_seconds = 0.0
        self.observer_seconds = 0.0
        self.recorded_history = []
        self.gradient_stats = {}

    def _observe(self, callback, *args):
        if callback is None: return
        start = sync_time()
        callback(*args)
        self.observer_seconds += sync_time()-start

    def _generate_image(self, current_latents, prompt):
        self._phase_start = sync_time()
        self._observer_start = self.observer_seconds
        return self.model.apply(latents=current_latents, prompt=prompt,
            num_inference_steps=self.n_inference_steps, num_images_per_prompt=len(current_latents))

    def _capture_initial(self, image, current_latents, prompt, multi_apply_fn):
        result = super()._capture_initial(image, current_latents, prompt, multi_apply_fn)
        self._observe(self.on_baseline, result[0])
        return result

    def _record_iteration(self, iteration, rewards, total_loss, return_history, iteration_history):
        if not math.isfinite(float(total_loss.detach())):
            raise RuntimeError(f'Non-finite official loss at iteration {iteration}; generation stopped.')
        super()._record_iteration(iteration, rewards, total_loss, return_history, iteration_history)
        self.iteration = iteration
        self._current_row = dict(iteration=iteration, evaluated_candidates=iteration+1,
            updates_before_evaluation=iteration, hpsv2_1_objective=rewards['HPS'],
            dpp_raw=rewards['diversity_dpp'], reward_loss=rewards['total'],
            loss_with_regularization=float(total_loss.detach()))
        denominator = float(NUM_IMAGES*torch.log(torch.tensor(2.0, dtype=torch.float32)))
        self._current_row['dpp_normalized'] = rewards['diversity_dpp']/denominator

    def _update_best(self, image, latents, rewards, total_reward_loss,
                     best_loss, best_image, best_rewards, best_latents):
        result = super()._update_best(image, latents, rewards, total_reward_loss,
                                      best_loss, best_image, best_rewards, best_latents)
        if result[-1]: self.best_iteration = self.iteration
        self._current_row['best_iteration'] = self.best_iteration
        self.optimization_seconds += (sync_time()-self._phase_start
                                      -(self.observer_seconds-self._observer_start))
        if self.iteration+1 in SAVE_BUDGETS:
            self._observe(self.on_budget, self, result[1], result[3], result[0])
        if self.iteration == self.n_iters-1:
            self._finish_iteration()
        return result

    def capture_noise_gradient(self, gradient):
        # Returning None leaves the gradient unchanged. Abort before SGD if it is invalid.
        if not torch.isfinite(gradient).all():
            raise RuntimeError(f'Non-finite gradient in official backward at iteration {self.iteration}. '
                               'No VJP scaling, retry or replacement gradient is applied.')
        self.gradient_stats = dict(gradient_norm_before_clip=float(gradient.float().norm()),
            gradient_abs_max=float(gradient.float().abs().max()),
            gradient_nonzero_fraction=float((gradient!=0).float().mean()))

    def _step_optimizer(self, total_loss, latents, optimizer, image, is_best_image, iteration, restored_mask=None):
        before = latents.detach().clone()
        start = sync_time()
        super()._step_optimizer(total_loss, latents, optimizer, image, is_best_image,
                                iteration, restored_mask=restored_mask)
        self.optimization_seconds += sync_time()-start
        if not torch.isfinite(latents).all():
            raise RuntimeError(f'Non-finite noise after the official SGD step at iteration {iteration}.')
        delta = latents.detach().float()-before.float()
        self._current_row.update(self.gradient_stats,
            noise_update_l2=float(delta.norm()),
            noise_changed_fraction=float((latents.detach()!=before).float().mean()))
        self._finish_iteration()

    def _finish_iteration(self):
        self._current_row['cumulative_optimization_seconds'] = self.optimization_seconds
        self.recorded_history.append(dict(self._current_row))
        self._observe(self.on_iteration, self)

def check_optimizer_gradient(optimizer, args, kwargs):
    for group in optimizer.param_groups:
        for parameter in group['params']:
            if parameter.grad is None or not torch.isfinite(parameter.grad).all():
                raise RuntimeError('Missing or non-finite gradient before the official optimiser step.')

def generation_environment():
    versions = {}
    for package in ['torch', 'torchvision', 'diffusers', 'transformers', 'tokenizers', 'huggingface-hub',
                    'accelerate', 'hpsv2', 'numpy', 'Pillow', 'timm']:
        versions[package] = importlib.metadata.version(package)
    return dict(python=sys.version, packages=versions, cuda=torch.version.cuda,
                gpu=torch.cuda.get_device_name() if torch.cuda.is_available() else None)

def verified_completed_group(root, path, config_hash, environment_hash):
    if not path.is_file(): return None
    saved = json.loads(path.read_text())
    if saved.get('status') != 'complete': return None
    if saved.get('config_sha256') != config_hash or saved.get('environment_sha256') != environment_hash:
        raise RuntimeError('Completed prompt belongs to a different configuration or environment. Use a new RUN_NAME.')
    expected={(name,i) for name in ['baseline']+[f'budget_{b:04d}' for b in SAVE_BUDGETS] for i in range(NUM_IMAGES)}
    actual=[(r['local_condition'],r['base_index']) for r in saved['records']]
    if len(actual)!=len(expected) or set(actual)!=expected:
        raise RuntimeError(f'Completed group has incomplete image coverage: {path}')
    if (saved['prompt_id']!=path.parent.parent.name or saved['init_name']!=path.parent.name
            or any(r['prompt_id']!=saved['prompt_id'] or r['init_name']!=saved['init_name'] for r in saved['records'])):
        raise RuntimeError(f'Completed group identity mismatch: {path}')
    for artifact in saved['artifacts']:
        local = root/artifact['relative_path']
        if not local.is_file() or file_sha(local)!=artifact['sha256']:
            raise RuntimeError(f'Completed noise or timing artifact missing or changed: {local}')
    for row in saved['records']:
        image = root/row['relative_image_path']
        if not image.is_file() or file_sha(image) != row['image_hash']:
            raise RuntimeError(f'Completed image missing or changed: {image}. Use a new RUN_NAME or restore the file.')
    return saved

def export_pil_images(root, item, init_name, images, local_condition, budget=None):
    condition = f"{item['prompt_id']}__{init_name}__{local_condition}"
    directory = root/'images'/condition
    directory.mkdir(parents=True, exist_ok=True)
    rows = []
    for i,image in enumerate(images):
        path = directory/f'{i:02d}.png'
        image.save(path)
        rows.append(dict(prompt_id=item['prompt_id'], prompt=item['prompt'], category=item['category'],
            init_name=init_name, local_condition=local_condition, condition=condition,
            baseline_condition=f"{item['prompt_id']}__{init_name}__baseline", base_index=i,
            seed=item['batch_seed']+i, image_path=str(path),
            relative_image_path=str(path.relative_to(root)), image_hash=file_sha(path), budget_evaluations=budget))
    if len(rows) != NUM_IMAGES: raise RuntimeError('The official trainer returned an unexpected image count.')
    return rows

def optimise_official_group(root, item, init_name, base, config_hash, environment_hash):
    directory = root/'prompts'/item['prompt_id']/init_name
    directory.mkdir(parents=True, exist_ok=True)
    complete = directory/'group.json'
    saved = verified_completed_group(root, complete, config_hash, environment_hash)
    if saved:
        print('SKIP completed group:', item['prompt_id'], init_name, flush=True)
        return saved
    group_start = sync_time()
    torch.cuda.reset_peak_memory_stats()
    fp32, converted = make_initial_noise(item, init_name, base)
    latents = torch.nn.Parameter(converted, requires_grad=True)
    optimizer = DIVGEN.get_optimizer('sgd', latents, LR)
    encoding_start = sync_time()
    generation_adapter.prepare(item)
    encoding_seconds = sync_time()-encoding_start
    setup_seconds = sync_time()-group_start
    initial = latents.detach().cpu().clone()
    initialization = initial_precision_report(base, fp32, converted)
    initialization.update(spec=INIT_CONFIGS[init_name],
        sample_seeds=[item['batch_seed']+i for i in range(NUM_IMAGES)],
        rollout_seeds=list(generation_adapter.seeds), scheduler_rng='per-image CPU generator reset per candidate',
        note='Shared generation applies scheduler.init_noise_sigma once to the unscaled FP16 noise.')
    torch.save(dict(initial_noise_fp32=fp32, initial_noise=initial), directory/'initialization.pt')
    atomic_json(directory/'initialization.json', initialization)
    records, budgets = [], []
    def baseline(images):
        records.extend(export_pil_images(root,item,init_name,images,'baseline'))
    def budget(trainer, best_images, best_latents, best_loss):
        evaluations = trainer.iteration+1
        selected = best_latents.detach().cpu().clone()
        local = f'budget_{evaluations:04d}'
        peak_allocated = torch.cuda.max_memory_allocated()/2**30
        peak_reserved = torch.cuda.max_memory_reserved()/2**30
        start = sync_time()
        pil_images = DIVGEN.LatentNoiseTrainer._tensor_to_pil(best_images)
        records.extend(export_pil_images(root,item,init_name,pil_images,local,evaluations))
        export_seconds = sync_time()-start
        torch.save(dict(initial_noise=initial, selected_noise=selected,
                        selected_iteration=trainer.best_iteration), directory/f'noise_{evaluations:04d}.pt')
        relative = ((selected.float()-initial.float()).flatten(1).norm(dim=1)
                    /initial.float().flatten(1).norm(dim=1)).max().item()
        budgets.append(dict(prompt_id=item['prompt_id'], category=item['category'], init_name=init_name,
            local_condition=local, evaluations=evaluations, updates=evaluations-1,
            selected_iteration=trainer.best_iteration, selected_reward_loss=best_loss,
            max_relative_edit=relative, prompt_encoding_seconds=encoding_seconds,
            prompt_setup_seconds=setup_seconds, optimization_seconds=trainer.optimization_seconds,
            warm_start_compute_seconds=setup_seconds+trainer.optimization_seconds,
            final_export_seconds=export_seconds,
            warm_start_method_seconds=setup_seconds+trainer.optimization_seconds+export_seconds,
            peak_cuda_allocated_gib=peak_allocated, peak_cuda_reserved_gib=peak_reserved,
            timing_scope='shared generation and official reward/backward; setup includes prompt encoding; budgets are trajectory prefixes'))
    def iteration(trainer):
        history = pd.DataFrame(trainer.recorded_history)
        history.insert(0,'init_name',init_name)
        history.insert(0,'prompt_id',item['prompt_id'])
        history.to_csv(directory/'history.csv',index=False)
        row = trainer.recorded_history[-1]
        if trainer.iteration%LOG_EVERY == 0 or trainer.iteration==N_EVALUATIONS-1:
            print(f"{item['prompt_id']} / {init_name} | {trainer.iteration+1}/{N_EVALUATIONS} candidates | "
                  f"HPS {row['hpsv2_1_objective']:.5f} | DPP {row['dpp_normalized']:.5f} | "
                  f"best {trainer.best_iteration} | {trainer.optimization_seconds:.1f}s",flush=True)
    trainer = RecordedDivGenTrainer(reward_losses=[hps_reward],model=generation_adapter,
        n_iters=N_EVALUATIONS,n_inference_steps=NUM_STEPS,seed=SCHEDULER_SEED,
        regularize=True,regularization_weight=REG_WEIGHT,grad_clip=GRAD_CLIP,
        log_metrics=False,device=torch.device(DEVICE),diversity_objectives=[dpp_objective],
        diversity_thresholds={},all_reward_losses=[hps_reward],all_diversity_objectives=[dpp_objective],
        on_baseline=baseline,on_budget=budget,on_iteration=iteration)
    gradient_hook = latents.register_hook(trainer.capture_noise_gradient)
    step_hook = optimizer.register_step_pre_hook(check_optimizer_gradient)
    try:
        train_start = sync_time()
        output = trainer.train(latents,item['prompt'],optimizer,None,None,return_history=True,
            skip_final_metrics_log=True)
        trainer_wall = sync_time()-train_start
        pd.DataFrame(output[4]).to_csv(directory/'official_history.csv',index=False)
        # End-to-end warm time includes initialisation, the complete official call and final output writing.
        end_to_end = sync_time()-group_start
        budgets[-1]['warm_start_end_to_end_seconds'] = end_to_end
        pd.DataFrame(budgets).to_csv(directory/'timing.csv',index=False)
        artifacts = [directory/'initialization.pt',directory/'initialization.json',
                     directory/'history.csv',directory/'official_history.csv',directory/'timing.csv']
        artifacts += [directory/f'noise_{b:04d}.pt' for b in SAVE_BUDGETS]
        result = dict(status='complete',prompt_id=item['prompt_id'],init_name=init_name,
            config_sha256=config_hash,environment_sha256=environment_hash,
            initialization=initialization,records=records,budgets=budgets,
            artifacts=[dict(relative_path=str(p.relative_to(root)),sha256=file_sha(p)) for p in artifacts],
            official_trainer_wall_seconds=trainer_wall,observer_seconds=trainer.observer_seconds,
            warm_start_end_to_end_seconds=end_to_end)
        atomic_json(complete,result)
        (directory/'failure.json').unlink(missing_ok=True)
        return result
    except BaseException as exc:
        atomic_json(directory/'failure.json',dict(error=repr(exc),prompt_id=item['prompt_id'],
            init_name=init_name,iteration=trainer.iteration,memory=memory_snapshot(),
            execution='official end-to-end backward; no VJP or numerical fallback',
            instruction='This unfinished group restarts from its initial noise. Completed groups are reusable.'))
        raise
    finally:
        gradient_hook.remove();step_hook.remove()
        hps_reward.clear_text_cache()
        # DINO is absent from the optimiser, as upstream. Discard unused parameter gradients between groups.
        dino_backend.dino_model.zero_grad(set_to_none=True)
        del trainer,optimizer,latents
        gc.collect();torch.cuda.empty_cache()

def generate_batch():
    if PROMPT_LIMIT is not None and not (isinstance(PROMPT_LIMIT,int) and 1<=PROMPT_LIMIT<=50):
        raise ValueError('PROMPT_LIMIT must be None or an integer from 1 to 50.')
    prompts = GENEVAL50 if PROMPT_LIMIT is None else GENEVAL50[:PROMPT_LIMIT]
    config = experiment_config(prompts)
    config_hash = object_sha(config)
    environment = generation_environment()
    environment_hash = object_sha(environment)
    name = generation_run_name(len(prompts))
    if Path(name).name!=name:raise ValueError('RUN_NAME must be a single directory name.')
    root = (OUTPUT_ROOT/name).resolve()
    manifest_path = root/'manifest.json'
    if root.exists() and any(root.iterdir()) and not manifest_path.is_file():
        raise FileExistsError('Non-empty directory has no generation manifest. Select a new RUN_NAME.')
    if manifest_path.is_file():
        prior = json.loads(manifest_path.read_text())
        if prior.get('config_sha256')!=config_hash or prior.get('environment_sha256')!=environment_hash:
            raise RuntimeError('Run settings or environment differ. Select a new RUN_NAME.')
        if not RESUME:raise FileExistsError('Run exists and RESUME is False.')
    root.mkdir(parents=True,exist_ok=True)
    # Remember this run across kernel changes; incomplete runs cannot be evaluated.
    atomic_json(OUTPUT_ROOT/'latest_generation.json', {'relative_run_dir': name})
    manifest = dict(method='divgen_sdxl_geneval50_two_init',status='running',project_root=str(PROJECT_ROOT),
        cache_dir=CACHE_DIR,config=config,config_sha256=config_hash,environment=environment,
        environment_sha256=environment_hash,prompts=prompts,records=[])
    completed,pending=[],[]
    for index,item in enumerate(prompts):
        names = list(INIT_CONFIGS) if index%2==0 else list(reversed(INIT_CONFIGS))
        for init_name in names:
            group = verified_completed_group(root,root/'prompts'/item['prompt_id']/init_name/'group.json',config_hash,environment_hash)
            if group:completed.append(group)
            else:pending.append((item,init_name))
    if pending:
        atomic_json(manifest_path,manifest)
        try:
            print('Generation directory:',root,flush=True)
            print('Pending groups:',len(pending),f'| Official trainer | SDXL-Turbo: {NUM_STEPS} denoising step(s)',flush=True)
            setup = load_official_models()
            atomic_json(root/f"setup_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')}.json",setup)
            for item,init_name in pending:
                base = shared_base_white(item)
                directory = root/'prompts'/item['prompt_id']
                directory.mkdir(parents=True,exist_ok=True)
                path = directory/'shared_base_white.pt'
                if path.exists():
                    if not torch.equal(torch.load(path,map_location='cpu',weights_only=True),base):
                        raise RuntimeError('Shared Gaussian samples differ from the saved run.')
                else:torch.save(base,path)
                completed.append(optimise_official_group(root,item,init_name,base,config_hash,environment_hash))
        finally:
            release_models()
            print('Generation models released. Memory:',memory_snapshot(),flush=True)
    else:print('All prompt groups are complete; no models will be loaded.',flush=True)
    order={p['prompt_id']:i for i,p in enumerate(prompts)}
    completed.sort(key=lambda g:(order[g['prompt_id']],list(INIT_CONFIGS).index(g['init_name'])))
    for item in prompts:
        pair=[g['initialization'] for g in completed if g['prompt_id']==item['prompt_id']]
        if len(pair)!=2 or pair[0]['base_white_sha256']!=pair[1]['base_white_sha256'] or pair[0]['rollout_seeds']!=pair[1]['rollout_seeds']:
            raise RuntimeError('The two initialisations are not correctly paired.')
    manifest.update(records=[r for g in completed for r in g['records']],status='complete',paired_initialization_verified=True)
    pd.DataFrame(manifest['records']).to_csv(root/'images.csv',index=False)
    pd.DataFrame([r for g in completed for r in g['budgets']]).to_csv(root/'timing_per_prompt.csv',index=False)
    atomic_json(manifest_path,manifest)
    print('Complete. Switch to the metrics kernel and start Section 7. RUN_TO_SCORE =',repr(str(root)),flush=True)
    return root

assert RecordedDivGenTrainer.train is DIVGEN.LatentNoiseTrainer.train


## 6. Generate the images

Run this cell in the generation kernel. The single batch contains pink and whitefloor groups; execution order alternates across prompts. Monitor `history.csv` for HPS, DPP, best iteration, gradient norms and actual noise changes. `official_history.csv` preserves the trainer's returned history. A failed group writes `failure.json`.


In [ ]:
RUN_DIR = generate_batch()


## 7. Select saved images for evaluation

Switch to the existing **metrics kernel** and start here when images have already been generated. This section is independent of all generation cells. With `RUN_TO_SCORE=None`, the cell selects the last run chosen by this notebook, including after a kernel change. Check its printed denoising step count. Set a folder name or full path in `RUN_TO_SCORE` to evaluate a different run. Missing, incomplete or modified results stop evaluation and direct you back to generation; evaluation never generates images.

Core metrics use `.venv-metrics`. MANIQA uses `.venv-maniqa` through a subprocess, while the notebook remains on the metrics kernel. Set `RUN_MANIQA=False` if that environment is not available; it can be added later without regenerating images. Each metric runs in a separate process and writes a log under its evaluation directory.

When using a custom `OUTPUT_ROOT`, also set `RESULTS_ROOT` here, or supply the full run path in `RUN_TO_SCORE`.


In [ ]:
from pathlib import Path
from collections import Counter
import hashlib, json, os, subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Evaluation is independent of every generation cell.
PROJECT_ROOT = None
def resolve_evaluation_project_root(value=None):
    cwd=Path.cwd().resolve()
    candidates=[Path(value).expanduser().resolve()] if value else [cwd,*cwd.parents]
    for candidate in candidates:
        if ((candidate/'noise_init/metric_runner.py').is_file()
                and (candidate/'pc_specific_psd/compat_metrics.py').is_file()): return candidate
    raise FileNotFoundError('Set PROJECT_ROOT to the project root containing noise_init and pc_specific_psd.')

PROJECT_ROOT=resolve_evaluation_project_root(PROJECT_ROOT)
RESULTS_ROOT=PROJECT_ROOT/'denoising_optimization/outputs/divgen_geneval50'
RUN_TO_SCORE=None  # Last selected run; alternatively, specify a run folder or full path.
CORE_PYTHON=PROJECT_ROOT/'.venv-metrics/bin/python'
MANIQA_PYTHON=PROJECT_ROOT/'.venv-maniqa/bin/python'
EVALUATION_TAG='evaluation_v1'
CORE_METRICS=['clip','hpsv3','dreamsim','lpips','vendi_clip']
RUN_MANIQA=True
DEVICE='cuda'
REFERENCE_METRIC_MANIFEST=None

def verify_existing_generation(results_root, run_to_score):
    """Validate both initialisations and their saved images without loading a generator."""
    if run_to_score is None:
        pointer = Path(results_root)/'latest_generation.json'
        if not pointer.is_file():
            raise FileNotFoundError('No generation run has been selected. Run Sections 1-6 first, '
                                    'or set RUN_TO_SCORE to an existing run directory.')
        run_to_score = json.loads(pointer.read_text())['relative_run_dir']
    if not str(run_to_score).strip(): raise ValueError('Set RUN_TO_SCORE explicitly.')
    target=Path(run_to_score).expanduser()
    root=(target if target.is_absolute() else Path(results_root)/target).resolve()
    if not root.is_dir():
        raise FileNotFoundError(f'Generation folder does not exist: {root}. Set RUN_TO_SCORE correctly or run Sections 1-6 first.')
    path=root/'manifest.json'
    if not path.is_file(): raise FileNotFoundError(f'Missing {path}. Select the generation batch root.')
    manifest=json.loads(path.read_text())
    if manifest.get('status')!='complete':
        raise ValueError('Generation is incomplete. Resume Sections 1-6 in the generation kernel before evaluation.')
    if manifest.get('method') not in ('divgen_sdxl_geneval50_two_init', 'divgen_sdxl4_geneval50_two_init'):
        raise ValueError('Expected the paired pink/whitefloor DivGen batch manifest.')
    config=manifest['config']; prompts=manifest['prompts']; rows=manifest['records']
    payload=json.dumps(config,sort_keys=True,separators=(',', ':')).encode()
    if hashlib.sha256(payload).hexdigest()!=manifest.get('config_sha256') or config['prompts']!=prompts:
        raise ValueError('Generation configuration or prompt manifest changed.')
    names=list(config['initializations'])
    if names!=['pink','whitefloor'] or not manifest.get('paired_initialization_verified'):
        raise ValueError('Both paired initialisations must be complete and verified.')
    if config['num_inference_steps'] not in (1, 4) or config['num_images']!=4:
        raise ValueError('Expected one or four denoising steps and four images per prompt.')
    ids=[p['prompt_id'] for p in prompts]
    if not ids or len(set(ids))!=len(ids): raise ValueError('Missing or duplicate prompt IDs.')
    conditions=['baseline']+[f'budget_{b:04d}' for b in config['save_budgets']]
    expected={(pid,name,c,i) for pid in ids for name in names for c in conditions for i in range(4)}
    actual=[(r['prompt_id'],r['init_name'],r['local_condition'],r['base_index']) for r in rows]
    if len(actual)!=len(expected) or set(actual)!=expected:
        raise ValueError('Incomplete or duplicate prompt / initialisation / condition / image coverage.')
    by_id={p['prompt_id']:p for p in prompts}
    for row in rows:
        item=by_id[row['prompt_id']]
        if (row['prompt']!=item['prompt'] or row['category']!=item['category']
                or row['seed']!=item['batch_seed']+row['base_index']):
            raise ValueError('Image metadata differs from its prompt or sample seed.')
        expected_condition=f"{row['prompt_id']}__{row['init_name']}__{row['local_condition']}"
        expected_baseline=f"{row['prompt_id']}__{row['init_name']}__baseline"
        if row['condition']!=expected_condition or row['baseline_condition']!=expected_baseline:
            raise ValueError('Condition or same-seed baseline pairing is incorrect.')
        image=(root/row['relative_image_path']).resolve()
        if not image.is_relative_to(root) or not image.is_file():
            raise FileNotFoundError(f'Missing image inside the selected generation run: {image}')
        if hashlib.sha256(image.read_bytes()).hexdigest()!=row['image_hash']:
            raise RuntimeError(f'Image changed after generation: {image}')
    for item in prompts:
        groups=[]
        for name in names:
            group=json.loads((root/'prompts'/item['prompt_id']/name/'group.json').read_text())
            if (group['status']!='complete' or group['config_sha256']!=manifest['config_sha256']
                    or group['init_name']!=name or group['prompt_id']!=item['prompt_id']):
                raise ValueError('Prompt/initialisation completion record does not match the batch.')
            groups.append(group['initialization'])
        if groups[0]['base_white_sha256']!=groups[1]['base_white_sha256'] or groups[0]['rollout_seeds']!=groups[1]['rollout_seeds']:
            raise ValueError('Paired initialisations used different Gaussian samples or scheduler random draws.')
    return root,manifest

if Path(EVALUATION_TAG).name!=EVALUATION_TAG or EVALUATION_TAG=='metrics':
    raise ValueError('EVALUATION_TAG must be a single directory name other than metrics.')
RUN_DIR,GENERATION_MANIFEST=verify_existing_generation(RESULTS_ROOT,RUN_TO_SCORE)
records=GENERATION_MANIFEST['records']
prompt_ids=[p['prompt_id'] for p in GENERATION_MANIFEST['prompts']]
EVAL_ROOT=RUN_DIR/EVALUATION_TAG
EVAL_ROOT.mkdir(exist_ok=True)
EVALUATED_NUM_STEPS = GENERATION_MANIFEST['config']['num_inference_steps']
print('Validated generation run:',RUN_DIR)
print('Denoising steps:', EVALUATED_NUM_STEPS)
print('Prompts:',len(prompt_ids),'| Initialisations: pink and whitefloor | Images:',len(records))
print('Saved evaluation budgets:',GENERATION_MANIFEST['config']['save_budgets'])
print('HPS reduction:',GENERATION_MANIFEST['config']['hps_reduction'])
print('Core interpreter:',CORE_PYTHON)
print('MANIQA interpreter:',MANIQA_PYTHON,'| Enabled:',RUN_MANIQA)
print('Evaluation root:',EVAL_ROOT)


## 8. Prepare the evaluation worker

The worker uses the same metric definitions as the FloorRefine notebook.


In [ ]:
METRICS_WORKER_SOURCE = '\n"""Standalone metrics worker embedded in the notebook; no generator imports."""\nfrom __future__ import annotations\nimport argparse, csv, gc, hashlib, importlib.metadata, json, sys, traceback, random\nfrom dataclasses import dataclass\nfrom itertools import combinations\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom PIL import Image\n\ndef file_hash(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\ndef sha256_text(*values):\n    return hashlib.sha256(json.dumps(values,sort_keys=True,default=str).encode()).hexdigest()\ndef version(name):\n    try: return importlib.metadata.version(name)\n    except importlib.metadata.PackageNotFoundError: return \'not-installed\'\n\ndef kernel_scores(vectors, compute_vendi=True):\n    # Exactly the project\'s kernel: already-normalized float32 CLIP embeddings;\n    # do not introduce a second normalization or float64 conversion.\n    v=np.stack(vectors)\n    K=v@v.T; K=(K+K.T)/2\n    if not np.allclose(np.diag(K),1.0,atol=1e-4):\n        raise RuntimeError(\'Vendi kernel diagonal is not one\')\n    value=None\n    if compute_vendi:\n        from vendi_score import vendi\n        value=float(vendi.score_K(K))\n    sign,ld=np.linalg.slogdet(K+1e-6*np.eye(len(v)))\n    if sign<=0: raise RuntimeError(\'Non-positive regularized kernel\')\n    return K,value,float(ld)\n\ndef basic_image_stats(path):\n    a=np.asarray(Image.open(path).convert(\'L\'),dtype=np.float64)/255.\n    dy,dx=np.gradient(a)\n    lap=a[2:,1:-1]+a[:-2,1:-1]+a[1:-1,2:]+a[1:-1,:-2]-4*a[1:-1,1:-1]\n    return {\'edge_gradient_mean\':float(np.sqrt(dx*dx+dy*dy).mean()),\'laplacian_variance\':float(lap.var())}\n\ndef read_csv(path):\n    if not path.exists(): return []\n    with path.open(newline=\'\',encoding=\'utf-8\') as f: return list(csv.DictReader(f))\ndef write_csv(path,rows,default_fields):\n    fields=list(dict.fromkeys(default_fields+[k for row in rows for k in row]))\n    tmp=path.with_suffix(\'.tmp\')\n    with tmp.open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,fieldnames=fields); writer.writeheader(); writer.writerows(rows)\n    tmp.replace(path)\n\ndef run(args):\n    root=Path(args.run_dir).resolve(); meta=json.loads((root/\'manifest.json\').read_text(encoding=\'utf-8\'))\n    out=Path(args.output_dir).resolve(); out.mkdir(parents=True,exist_ok=True)\n    if out == root/\'metrics\': raise ValueError(\'Use a new output directory; old metric results are protected.\')\n    groups={}\n    for row in meta[\'records\']:\n        # Prefer relative layout, so moving the whole run folder still works.\n        local = root/row[\'relative_image_path\'] if row.get(\'relative_image_path\') else root/\'images\'/row[\'condition\']/Path(row[\'image_path\']).name\n        row=dict(row,image_path=str(local if local.exists() else Path(row[\'image_path\'])))\n        if not Path(row[\'image_path\']).is_file(): raise FileNotFoundError(row[\'image_path\'])\n        if file_hash(row[\'image_path\'])!=row[\'image_hash\']: raise RuntimeError(\'Image changed after generation: \'+row[\'image_path\'])\n        groups.setdefault(row[\'condition\'],[]).append(row)\n    for rows in groups.values():\n        rows.sort(key=lambda r:r[\'base_index\'])\n        assert len(rows)==4 and len({r[\'base_index\'] for r in rows})==4\n    device=torch.device(args.device)\n    config={\'quality_metrics\':{\'clip\':{\'enabled\':True,\'checkpoint\':\'openai/clip-vit-large-patch14\'},\'hpsv3\':{\'enabled\':True}},\n            \'diversity_metrics\':{\'lpips\':{\'enabled\':True,\'backbone\':\'alex\'},\'dreamsim\':{\'enabled\':True},\'vendi_clip\':{\'enabled\':True}},\'cache_dir\':None}\n    project=Path(args.project_root or meta.get(\'project_root\') or \'\').expanduser().resolve()\n    requested={m.strip() for m in args.metrics.split(\',\') if m.strip()}\n    runner=None\n    if requested <= {\'maniqa\', \'basic\'}:\n        backend=\'isolated_pyiqa_maniqa\'; source_sha=None\n    else:\n        if \'maniqa\' in requested:\n            raise ValueError(\'MANIQA must run in its own interpreter and output directory.\')\n        if not (project/\'pc_specific_psd/compat_metrics.py\').is_file():\n            raise FileNotFoundError(\'Set PROJECT_ROOT to the repository root.\')\n        sys.path.insert(0,str(project))\n        from pc_specific_psd.compat_metrics import MetricRunner\n        import inspect\n        source=Path(inspect.getfile(MetricRunner)).resolve()\n        if source != (project/\'noise_init/metric_runner.py\').resolve():\n            raise RuntimeError(\'MetricRunner imported from another repository.\')\n        backend=\'project_noise_init_MetricRunner\'; source_sha=file_hash(source)\n        config[\'cache_dir\']=meta.get(\'cache_dir\')\n        runner=MetricRunner(config,device)\n    info={\'backend\':backend,\'metric_runner_sha256\':source_sha,\'device\':str(device),\n          \'versions\':{x:version(x) for x in [\'torch\',\'torchvision\',\'transformers\',\'tokenizers\',\'huggingface-hub\',\'peft\',\'trl\',\'numpy\',\'Pillow\',\'timm\',\'lpips\',\'dreamsim\',\'vendi-score\',\'hpsv3\',\'image-reward\',\'pyiqa\']},\n          \'clip_checkpoint\':\'openai/clip-vit-large-patch14\',\'dino_checkpoint\':\'facebook/dinov2-base\',\n          \'lpips_resize\':[256,256],\'logdet_definition\':\'log det(K_cosine + 1e-6 I)\',\n          \'vendi_definition\':\'exp(-sum eigen(K/n) log eigen(K/n))\',\n          \'basic_stats\':\'grayscale [0,1], numpy gradient; interior five-point Laplacian variance\'}\n    info.update(python=sys.executable, python_version=sys.version,\n        packages={d.metadata[\'Name\']:d.version for d in importlib.metadata.distributions() if d.metadata[\'Name\']},\n        worker_sha256=file_hash(__file__), generation_manifest_sha256=file_hash(root/\'manifest.json\'),\n        maniqa_sampling=\'Per-image SHA256(image_hash|maniqa-v1) seed; official default MANIQA preprocessing and checkpoint\')\n    prior=out/\'metric_manifest.json\'\n    if prior.exists() and json.loads(prior.read_text()) != info:\n        raise RuntimeError(\'Environment, worker or inputs changed. Choose a new EVALUATION_TAG to avoid mixed versions.\')\n    (out/\'metric_manifest.json\').write_text(json.dumps(info,indent=2),encoding=\'utf-8\')\n    print(\'Metrics backend:\',backend,\'device:\',device,flush=True)\n    tables={name:read_csv(out/(name+\'.csv\')) for name in [\'per_image\',\'per_pair\',\'per_group\',\'paired_change\',\'metric_status\']}\n    defaults={\'per_image\':[\'condition\',\'base_index\',\'metric\',\'score\',\'task\'],\n              \'per_pair\':[\'condition\',\'left\',\'right\',\'metric\',\'score\',\'task\'],\n              \'per_group\':[\'condition\',\'metric\',\'score\',\'n\',\'std\',\'minimum\',\'maximum\',\'task\'],\n              \'paired_change\':[\'condition\',\'base_index\',\'metric\',\'score\',\'task\'],\n              \'metric_status\':[\'task\',\'status\',\'error\']}\n    def checkpoint():\n        for name,rows in tables.items():\n            for row in rows:\n                if row.get(\'condition\') in groups:\n                    origin = groups[row[\'condition\']][0]\n                    row.update(prompt_id=origin[\'prompt_id\'], category=origin[\'category\'],\n                               init_name=origin[\'init_name\'], local_condition=origin[\'local_condition\'])\n            write_csv(out/(name+\'.csv\'),rows,defaults[name])\n    def group_record(condition,metric,values,task):\n        a=np.array(values,float)\n        if not np.isfinite(a).all(): raise RuntimeError(\'Non-finite metric \'+metric)\n        return dict(condition=condition,metric=metric,score=float(a.mean()),n=len(a),std=float(a.std()),minimum=float(a.min()),maximum=float(a.max()),task=task)\n    enabled=list(dict.fromkeys(([] if requested <= {\'maniqa\'} else [\'basic\'])+[m.strip() for m in args.metrics.split(\',\') if m.strip()]))\n    allowed={\'basic\',\'clip\',\'hpsv3\',\'dreamsim\',\'lpips\',\'vendi_clip\',\'maniqa\'}\n    if set(enabled)-allowed: raise ValueError(\'Unknown metrics: \'+str(set(enabled)-allowed))\n    # Valid only for this worker, input image hashes/prompts, backend and versions.\n    signature=sha256_text(file_hash(__file__),info,\n        [{k:r.get(k) for k in [\'condition\',\'base_index\',\'prompt\',\'image_hash\',\'baseline_condition\']} for r in meta[\'records\']])\n    done_path=out/\'completed_tasks.json\'\n    done=json.loads(done_path.read_text()) if done_path.exists() else {}\n    for task in enabled:\n        cached=[r for r in tables[\'per_group\'] if r.get(\'task\')==task]\n        # Tables contain multiple tasks, so verify per-task row digests (not whole-file hashes).\n        cache_ok=(done.get(task,{}).get(\'signature\')==signature and\n                  done.get(task,{}).get(\'rows_digest\')==sha256_text({\n                    name:[{k:str(v) for k,v in r.items()} for r in data if r.get(\'task\')==task]\n                    for name,data in tables.items()}) and\n                  {r[\'condition\'] for r in cached}==set(groups))\n        if cache_ok:\n            print(\'SKIP completed metric:\',task,flush=True)\n            continue\n        print(\'START\',task,flush=True)\n        for memory_file in [\'memory.current\',\'memory.max\',\'memory.peak\',\'memory.events\']:\n            try: print(memory_file, (Path(\'/sys/fs/cgroup\')/memory_file).read_text().strip(), flush=True)\n            except OSError: pass\n        fresh={k:[] for k in tables}; extra_model=None\n        try:\n            if task in [\'basic\',\'clip\',\'hpsv3\',\'imagereward\',\'maniqa\',\'topiq\']:\n                if task==\'imagereward\':\n                    import ImageReward as RM\n                    extra_model=RM.load(\'ImageReward-v1.0\',device=str(device))\n                if task in [\'maniqa\',\'topiq\']:\n                    import pyiqa\n                    extra_model=pyiqa.create_metric(\'maniqa\' if task==\'maniqa\' else \'topiq_nr\',device=device)\n                for condition,rows in groups.items():\n                    collected={}\n                    for row in rows:\n                        p=Path(row[\'image_path\'])\n                        if task==\'basic\': values=basic_image_stats(p)\n                        elif task==\'clip\': values={\'clip_cosine\':runner.clip_cosine(p,row[\'prompt\'])}\n                        elif task==\'hpsv3\': values={\'hpsv3\':runner.hpsv3(p,row[\'prompt\'])}\n                        elif task==\'imagereward\':\n                            with torch.inference_mode(): val=float(extra_model.score(row[\'prompt\'],str(p)))\n                            values={\'imagereward\':val}\n                        else:\n                            # MANIQA may sample image crops. Seed each image independently of task order.\n                            sample_seed=int(hashlib.sha256((row[\'image_hash\']+\'|maniqa-v1\').encode()).hexdigest()[:8],16)\n                            random.seed(sample_seed); np.random.seed(sample_seed); torch.manual_seed(sample_seed)\n                            if torch.cuda.is_available(): torch.cuda.manual_seed_all(sample_seed)\n                            with torch.inference_mode(): val=float(extra_model(str(p)).item())\n                            values={task:val}\n                        for metric,value in values.items():\n                            if not np.isfinite(value): raise RuntimeError(\'Non-finite \'+metric)\n                            fresh[\'per_image\'].append(dict(condition=condition,base_index=row[\'base_index\'],metric=metric,score=value,task=task))\n                            collected.setdefault(metric,[]).append(value)\n                    for metric,vals in collected.items(): fresh[\'per_group\'].append(group_record(condition,metric,vals,task))\n            if task in [\'dreamsim\',\'lpips\']:\n                fn=runner.dreamsim_distance if task==\'dreamsim\' else runner.lpips_distance\n                metric=task+\'_mean_pair_distance\' if task==\'dreamsim\' else \'lpips_alex_mean_pair_distance\'\n                for condition,rows in groups.items():\n                    vals=[]\n                    for i,j in combinations(range(4),2):\n                        value=fn(Path(rows[i][\'image_path\']),Path(rows[j][\'image_path\'])); vals.append(value)\n                        fresh[\'per_pair\'].append(dict(condition=condition,left=i,right=j,metric=metric,score=value,task=task))\n                    fresh[\'per_group\'].append(group_record(condition,metric,vals,task))\n                    baseline_condition = rows[0].get(\'baseline_condition\', \'baseline\')\n                    if condition != baseline_condition:\n                        for i,row in enumerate(rows):\n                            value=fn(Path(row[\'image_path\']),Path(groups[baseline_condition][i][\'image_path\']))\n                            if not np.isfinite(value): raise RuntimeError(\'Non-finite paired distance\')\n                            fresh[\'paired_change\'].append(dict(condition=condition,baseline_condition=baseline_condition,base_index=i,metric=task+\'_distance_to_same_seed_baseline\',score=value,task=task))\n            if task in [\'clip\',\'vendi_clip\',\'dino\']:\n                if task==\'dino\':\n                    from transformers import AutoImageProcessor,AutoModel\n                    processor=AutoImageProcessor.from_pretrained(\'facebook/dinov2-base\')\n                    extra_model=AutoModel.from_pretrained(\'facebook/dinov2-base\').to(device).eval()\n                for condition,rows in groups.items():\n                    vectors=[]\n                    for row in rows:\n                        if task==\'dino\':\n                            inp=processor(images=Image.open(row[\'image_path\']).convert(\'RGB\'),return_tensors=\'pt\').to(device)\n                            with torch.inference_mode(): v=extra_model(**inp).last_hidden_state[:,0,:][0].float().cpu().numpy()\n                            v=v/max(float(np.linalg.norm(v)),1e-12)\n                        else: v=runner.clip_embedding(Path(row[\'image_path\']))\n                        vectors.append(v)\n                    kernel,vendi,logdet=kernel_scores(vectors,compute_vendi=task in [\'vendi_clip\',\'dino\'])\n                    name=\'dino\' if task==\'dino\' else \'clip\'\n                    if task in [\'vendi_clip\',\'dino\']:\n                        fresh[\'per_group\'].append(group_record(condition,\'vendi_\'+name,[vendi],task))\n                        fresh[\'per_group\'][-1][\'n\']=4\n                    if task in [\'clip\',\'dino\']:\n                        vals=[]\n                        for i,j in combinations(range(4),2):\n                            value=float(1-kernel[i,j]); vals.append(value)\n                            fresh[\'per_pair\'].append(dict(condition=condition,left=i,right=j,metric=name+\'_cosine_distance\',score=value,task=task))\n                        fresh[\'per_group\'].append(group_record(condition,name+\'_mean_cosine_distance\',vals,task))\n                        fresh[\'per_group\'].append(group_record(condition,name+\'_regularized_logdet\',[logdet],task))\n                        fresh[\'per_group\'][-1][\'n\']=4\n            # Atomic per-task replacement; incomplete tasks never masquerade as complete metrics.\n            for name in tables:\n                if name==\'metric_status\': continue\n                tables[name]=[r for r in tables[name] if r.get(\'task\')!=task]+fresh[name]\n            status={\'task\':task,\'status\':\'ok\',\'error\':\'\'}\n            print(\'OK\',task,flush=True)\n        except Exception as exc:\n            status={\'task\':task,\'status\':\'error\',\'error\':repr(exc)}\n            # Do not keep stale successful scores for a failed rerun of this task.\n            for name in tables:\n                if name!=\'metric_status\': tables[name]=[r for r in tables[name] if r.get(\'task\')!=task]\n            print(\'ERROR\',task,repr(exc),flush=True)\n            (out/(task+\'_error.txt\')).write_text(traceback.format_exc(),encoding=\'utf-8\')\n        finally:\n            extra_model=None\n            if \'inp\' in locals(): del inp\n            if \'processor\' in locals(): del processor\n            if runner is not None: runner.release_models()\n            gc.collect()\n            if device.type==\'cuda\': torch.cuda.empty_cache()\n        tables[\'metric_status\']=[r for r in tables[\'metric_status\'] if r.get(\'task\')!=task]+[status]\n        checkpoint()\n        # Normalize through CSV so resume compares identical string representations.\n        tables={name:read_csv(out/(name+\'.csv\')) for name in tables}\n        if status[\'status\']==\'ok\':\n            done[task]={\'signature\':signature,\'rows_digest\':sha256_text({\n                name:[{k:str(v) for k,v in r.items()} for r in data if r.get(\'task\')==task]\n                for name,data in tables.items()})}\n        else: done.pop(task,None)\n        tmp=done_path.with_suffix(\'.tmp\'); tmp.write_text(json.dumps(done,indent=2)); tmp.replace(done_path)\n    print(\'Complete. Check metric_status.csv for failed metrics.\',flush=True)\n    failed=[r for r in tables[\'metric_status\'] if r[\'task\'] in enabled and r[\'status\']!=\'ok\']\n    if failed: raise RuntimeError(\'Requested metric tasks failed: \'+str(failed))\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--run-dir\',required=True)\n    parser.add_argument(\'--project-root\',default=None)\n    parser.add_argument(\'--output-dir\',required=True)\n    parser.add_argument(\'--device\',default=\'cuda\')\n    parser.add_argument(\'--metrics\',default=\'clip,hpsv3,dreamsim,lpips,vendi_clip\')\n    run(parser.parse_args())\n\n'
WORKER_PATH=EVAL_ROOT/'isolated_metrics_worker.py'
WORKER_PATH.write_text(METRICS_WORKER_SOURCE,encoding='utf-8')
print('Worker:',WORKER_PATH)


## 9. Compute core metrics

Compute HPSv3, CLIP alignment, DreamSim, LPIPS-Alex, CLIP-Vendi, and basic image statistics. Valid completed tasks are skipped; results and logs are saved under `EVAL_ROOT/core`.


In [ ]:
def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def execute_metric(metric, python, output):
    python=Path(python)
    if not python.is_file(): raise FileNotFoundError(f'Interpreter missing: {python}')
    output.mkdir(parents=True, exist_ok=True)
    env=os.environ.copy()
    # Prevent the notebook environment from leaking import paths into the worker.
    for key in ['PYTHONPATH','PYTHONHOME']: env.pop(key,None)
    env.update(MPLBACKEND='Agg',PYTHONUNBUFFERED='1',PYTHONNOUSERSITE='1',
               OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2')
    cmd=[str(python),str(WORKER_PATH),'--run-dir',str(RUN_DIR),
         '--project-root',str(PROJECT_ROOT),'--output-dir',str(output),
         '--device',DEVICE,'--metrics',metric]
    print(f'Running {metric} with {python}',flush=True)
    with (output/(metric+'.log')).open('a',encoding='utf-8') as log:
        proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,bufsize=1,env=env)
        try:
            for line in proc.stdout:
                print(line,end='',flush=True); log.write(line); log.flush()
            code=proc.wait()
        except BaseException:
            proc.terminate()
            try: proc.wait(timeout=10)
            except subprocess.TimeoutExpired: proc.kill(); proc.wait()
            raise
        finally: proc.stdout.close()
    if code:
        raise RuntimeError(f'{metric} exited with {code}. See {output/(metric+".log")}. '
                           'For -9, compare cgroup memory.events oom_kill before/after. Images are unchanged.')
    status=pd.read_csv(output/'metric_status.csv',keep_default_na=False)
    row=status[status.task==metric]
    if len(row)!=1 or row.iloc[0]['status']!='ok':
        raise RuntimeError(f'{metric} did not complete successfully.')

for metric in CORE_METRICS:
    if metric not in ['clip','hpsv3','dreamsim','lpips','vendi_clip']:
        raise ValueError('Only the five original core metrics belong in CORE_METRICS.')
    execute_metric(metric,CORE_PYTHON,EVAL_ROOT/'core')


## 10. Compute MANIQA

Keep the metrics kernel selected. This cell runs MANIQA in its dedicated environment and saves the scores under `EVAL_ROOT/maniqa`.


In [ ]:
if RUN_MANIQA:
    execute_metric('maniqa',MANIQA_PYTHON,EVAL_ROOT/'maniqa')
else:
    print('MANIQA disabled; core scoring and summaries remain available.')


## 11. Inspect metric versions

Optionally set `REFERENCE_METRIC_MANIFEST` to compare package versions with another experiment.


In [ ]:
core_manifest=json.loads((EVAL_ROOT/'core/metric_manifest.json').read_text())
print('Core evaluator versions:')
display(pd.Series(core_manifest['versions'],name='current_version'))
print('Project metric code SHA256:',core_manifest['metric_runner_sha256'])
if REFERENCE_METRIC_MANIFEST:
    ref=json.loads(Path(REFERENCE_METRIC_MANIFEST).read_text())
    old=ref.get('versions') or ref.get('metric_versions') or {}
    if not old:
        print('No recognized version dictionary in the reference; inspect its schema manually.')
    else:
        comparison=pd.DataFrame([{'package':k,'reference':v,
            'current':core_manifest['versions'].get(k,core_manifest['packages'].get(k,'not-recorded'))}
            for k,v in old.items()])
        comparison['matches']=comparison.reference==comparison.current
        display(comparison)
        comparison.to_csv(EVAL_ROOT/'reference_version_comparison.csv',index=False)
    print('Package matches alone do not verify checkpoint, preprocessing or code identity.')
else:
    print('No reference metric manifest selected; cross-experiment version comparison skipped.')


## 12. Combine the results

Validated results are saved under `EVAL_ROOT/combined_with_maniqa`, or `combined_core` when MANIQA is disabled.


In [ ]:
def merge_evaluations():
    core=EVAL_ROOT/'core'
    roots=[(core,set(CORE_METRICS)|{'basic'})]
    if RUN_MANIQA: roots.append((EVAL_ROOT/'maniqa',{'maniqa'}))
    expected_conditions={r['condition'] for r in records}
    manifests=[]
    for root,tasks in roots:
        manifest=json.loads((root/'metric_manifest.json').read_text())
        if manifest['generation_manifest_sha256']!=sha256_file(RUN_DIR/'manifest.json'):
            raise RuntimeError('Generation manifest changed after scoring.')
        if manifest['worker_sha256']!=sha256_file(WORKER_PATH):
            raise RuntimeError('Worker changed after scoring; run the metric cells again.')
        status=pd.read_csv(root/'metric_status.csv',keep_default_na=False)
        for task in tasks:
            row=status[status.task==task]
            if len(row)!=1 or row.iloc[0]['status']!='ok':
                raise RuntimeError(f'Missing or failed task: {task} in {root}')
        manifests.append({'path':str(root/'metric_manifest.json'),'sha256':sha256_file(root/'metric_manifest.json')})
    folder='combined_with_maniqa' if RUN_MANIQA else 'combined_core'
    combined=EVAL_ROOT/folder
    combined.mkdir(exist_ok=True)
    tables={}
    for name in ['per_image','per_pair','per_group','paired_change','metric_status']:
        frames=[]
        for root,tasks in roots:
            frame=pd.read_csv(root/(name+'.csv'),keep_default_na=False)
            frames.append(frame[frame.task.isin(tasks)])
        merged=pd.concat(frames,ignore_index=True)
        keys={'per_image':['condition','base_index','metric','task'],
              'per_pair':['condition','left','right','metric','task'],
              'per_group':['condition','metric','task'],
              'paired_change':['condition','base_index','metric','task'],
              'metric_status':['task']}[name]
        if merged.duplicated(keys).any(): raise RuntimeError(f'Duplicate records in {name}')
        if name!='metric_status' and len(merged):
            if not np.isfinite(pd.to_numeric(merged.score,errors='coerce')).all():
                raise RuntimeError(f'Non-finite scores in {name}')
        tables[name]=merged
    required={'clip':'clip_cosine','hpsv3':'hpsv3','dreamsim':'dreamsim_mean_pair_distance',
              'lpips':'lpips_alex_mean_pair_distance','vendi_clip':'vendi_clip'}
    if RUN_MANIQA: required['maniqa']='maniqa'
    for task in CORE_METRICS+(['maniqa'] if RUN_MANIQA else []):
        group=tables['per_group']; selected=group[(group.task==task)&(group.metric==required[task])]
        if len(selected)!=len(expected_conditions) or set(selected.condition)!=expected_conditions:
            raise RuntimeError(f'Incomplete group coverage for {task}; no partial mean will be reported.')
    for name,frame in tables.items(): frame.to_csv(combined/(name+'.csv'),index=False)
    (combined/'evaluation_sources.json').write_text(json.dumps({
        'generation_manifest_sha256':sha256_file(RUN_DIR/'manifest.json'),
        'sources':manifests,'core_metrics':CORE_METRICS,'includes_maniqa':RUN_MANIQA},indent=2))
    return combined

metrics_dir=merge_evaluations()
print('Validated merged results:',metrics_dir)


## 13. Summarise quality, diversity and cost

Each prompt receives equal weight. Confidence intervals use 2,000 paired prompt-bootstrap resamples. They describe variation across the selected prompts, not uncertainty from repeated independent seed groups. MANIQA measures no-reference perceptual image quality; it is not a pure sharpness score. Like the FloorRefine summary, results are grouped by initialisation and compared with their own baseline. Q-D arrows connect unoptimised means to selected-candidate means, with one figure per saved budget. Tables include paired whitefloor-minus-pink differences and timing summaries.


In [ ]:
GROUP=pd.read_csv(metrics_dir/'per_group.csv')
lookup={row['condition']:row for row in GENERATION_MANIFEST['records']}
for column in ['prompt_id','category','init_name','local_condition']:
    GROUP[column]=GROUP.condition.map(lambda key:lookup[key][column])
GROUP.to_csv(metrics_dir/'per_prompt_scores.csv',index=False)
prompt_ids=[p['prompt_id'] for p in GENERATION_MANIFEST['prompts']]
INIT_NAMES=list(GENERATION_MANIFEST['config']['initializations'])
INIT_LABELS={name:spec['label'] for name,spec in GENERATION_MANIFEST['config']['initializations'].items()}
BUDGETS=GENERATION_MANIFEST['config']['save_budgets']
BOOTSTRAP_SEED,BOOTSTRAP_REPLICATES=20261023,2000
boot=np.random.default_rng(BOOTSTRAP_SEED).integers(0,len(prompt_ids),(BOOTSTRAP_REPLICATES,len(prompt_ids)))

def paired_mean_ci(values):
    values=np.asarray(values,dtype=float)
    if values.shape!=(len(prompt_ids),) or not np.isfinite(values).all():
        raise ValueError('Expected one finite value for every prompt.')
    low,high=np.quantile(values[boot].mean(1),[0.025,0.975])
    return float(values.mean()),float(low),float(high)

conditions=['baseline']+[f'budget_{b:04d}' for b in BUDGETS]
summaries,contrasts,details=[],[],[]
SCORE_TABLES={}
expected_columns=pd.MultiIndex.from_product([INIT_NAMES,conditions],names=['init_name','local_condition'])
for metric,part in GROUP.groupby('metric'):
    table=part.pivot(index='prompt_id',columns=['init_name','local_condition'],values='score').reindex(index=prompt_ids,columns=expected_columns)
    if not np.isfinite(table.to_numpy()).all(): raise RuntimeError(f'Incomplete metric: {metric}')
    SCORE_TABLES[metric]=table
    for name in INIT_NAMES:
        baseline=table[(name,'baseline')].to_numpy()
        for condition in conditions:
            after=table[(name,condition)].to_numpy()
            mean,low,high=paired_mean_ci(after)
            delta,delta_low,delta_high=paired_mean_ci(after-baseline)
            summaries.append(dict(metric=metric,init_name=name,condition=condition,n_prompts=len(prompt_ids),mean=mean,
                ci_low=low,ci_high=high,change_from_baseline=delta,change_ci_low=delta_low,change_ci_high=delta_high))
            for pid,b,a in zip(prompt_ids,baseline,after):
                details.append(dict(prompt_id=pid,init_name=name,condition=condition,metric=metric,baseline=b,score=a,change=a-b))
    for condition in conditions:
        difference,low,high=paired_mean_ci((table[('whitefloor',condition)]-table[('pink',condition)]).to_numpy())
        contrasts.append(dict(metric=metric,condition=condition,contrast='whitefloor minus pink',n_prompts=len(prompt_ids),
                              difference=difference,ci_low=low,ci_high=high))
SUMMARY=pd.DataFrame(summaries)
CONTRASTS=pd.DataFrame(contrasts)
SUMMARY.to_csv(metrics_dir/'summary_prompt_equal_weight.csv',index=False)
CONTRASTS.to_csv(metrics_dir/'paired_initialization_comparisons.csv',index=False)
pd.DataFrame(details).to_csv(metrics_dir/'per_prompt_delta.csv',index=False)
GROUP.groupby(['category','init_name','local_condition','metric']).score.agg(['mean','count']).to_csv(metrics_dir/'summary_by_category.csv')
KEY_METRICS=['hpsv3','clip_cosine','maniqa','dreamsim_mean_pair_distance','lpips_alex_mean_pair_distance','vendi_clip']
DISPLAY_METRICS={'hpsv3':'HPSv3','clip_cosine':'CLIP cosine','maniqa':'MANIQA',
    'dreamsim_mean_pair_distance':'DreamSim distance','lpips_alex_mean_pair_distance':'LPIPS-Alex distance','vendi_clip':'CLIP-Vendi'}
display(SUMMARY[SUMMARY.metric.isin(KEY_METRICS)])
print('Paired final-score contrasts; a larger optimisation gain alone does not imply a better final result.')
display(CONTRASTS[CONTRASTS.metric.isin(KEY_METRICS)])

TIMING=pd.read_csv(RUN_DIR/'timing_per_prompt.csv')
expected_timing={(pid,name,budget) for pid in prompt_ids for name in INIT_NAMES for budget in BUDGETS}
actual_timing=list(TIMING[['prompt_id','init_name','evaluations']].itertuples(index=False,name=None))
if len(actual_timing)!=len(expected_timing) or set(actual_timing)!=expected_timing:
    raise RuntimeError('Incomplete or duplicate prompt/initialisation/budget timings.')
cost_columns=['optimization_seconds','warm_start_compute_seconds','warm_start_method_seconds',
              'peak_cuda_allocated_gib','peak_cuda_reserved_gib']
if not np.isfinite(TIMING[cost_columns].to_numpy()).all(): raise RuntimeError('Non-finite timing or memory records.')
TIMING_SUMMARY=TIMING.groupby(['init_name','evaluations'])[cost_columns].agg(['mean','median','min','max'])
TIMING_SUMMARY.to_csv(metrics_dir/'timing_summary.csv')
display(TIMING_SUMMARY)
paired_cost=[]
for budget in BUDGETS:
    part=TIMING[TIMING.evaluations==budget]
    for metric in cost_columns:
        costs=part.pivot(index='prompt_id',columns='init_name',values=metric).reindex(prompt_ids)
        mean,low,high=paired_mean_ci((costs.whitefloor-costs.pink).to_numpy())
        paired_cost.append(dict(evaluations=budget,metric=metric,whitefloor_minus_pink=mean,ci_low=low,ci_high=high))
pd.DataFrame(paired_cost).to_csv(metrics_dir/'paired_timing_comparisons.csv',index=False)

def save_qd_arrow_figures(group, labels, initial_condition, selected_conditions, output_prefix, colors):
    """Each arrow starts at its method's own unoptimised baseline; one panel grid per budget."""
    from matplotlib.lines import Line2D
    qualities=[m for m in ['hpsv3','clip_cosine','maniqa'] if m in set(group.metric)]
    diversities=[m for m in ['dreamsim_mean_pair_distance','lpips_alex_mean_pair_distance','vendi_clip'] if m in set(group.metric)]
    if not qualities or not diversities: return
    averages=group.groupby(['init_name','local_condition','metric']).score.mean()
    for budget,selections in selected_conditions.items():
        fig,axes=plt.subplots(len(qualities),len(diversities),figsize=(5*len(diversities),3.6*len(qualities)),squeeze=False)
        for row,quality in enumerate(qualities):
            for col,diversity in enumerate(diversities):
                ax=axes[row,col]
                for name,label in labels.items():
                    before=initial_condition[name];after=selections[name]
                    x0=averages.loc[(name,before,diversity)];y0=averages.loc[(name,before,quality)]
                    x1=averages.loc[(name,after,diversity)];y1=averages.loc[(name,after,quality)]
                    color=colors[name]
                    ax.scatter(x0,y0,color=color,marker='o',facecolors='none',s=50,zorder=3)
                    ax.scatter(x1,y1,color=color,marker='^',s=60,zorder=4)
                    ax.annotate('',xy=(x1,y1),xytext=(x0,y0),
                        arrowprops=dict(arrowstyle='->',color=color,lw=1.8,shrinkA=4,shrinkB=5),zorder=2)
                for metric,setter in [(diversity,ax.set_xlim),(quality,ax.set_ylim)]:
                    values=averages.xs(metric,level='metric').to_numpy(dtype=float)
                    span=max(float(values.max()-values.min()),abs(float(values.mean()))*0.02,1e-5)
                    setter(float(values.min())-span*.18,float(values.max())+span*.18)
                ax.set(xlabel=DISPLAY_METRICS[diversity],ylabel=DISPLAY_METRICS[quality])
                ax.grid(alpha=.2)
        handles=[Line2D([0],[0],color=colors[name],lw=2,label=label) for name,label in labels.items()]
        handles += [Line2D([0],[0],color='black',marker='o',markerfacecolor='none',linestyle='None',label='Unoptimised'),
                    Line2D([0],[0],color='black',marker='^',linestyle='None',label='Selected candidate')]
        fig.suptitle(f'SDXL-Turbo, {EVALUATED_NUM_STEPS} denoising step(s) | {len(prompt_ids)} prompts | DivGen: {budget} evaluations ({budget-1} updates)',y=.995)
        fig.legend(handles=handles,loc='upper center',bbox_to_anchor=(.5,.965),ncol=2,frameon=False)
        fig.tight_layout(rect=(0,0,1,.88))
        stem=metrics_dir/f'{output_prefix}_eval{budget:04d}'
        fig.savefig(stem.with_suffix('.png'),dpi=160)
        fig.savefig(stem.with_suffix('.pdf'))
        plt.show();plt.close(fig)

save_qd_arrow_figures(GROUP,INIT_LABELS,{name:'baseline' for name in INIT_NAMES},
    {budget:{name:f'budget_{budget:04d}' for name in INIT_NAMES} for budget in BUDGETS},
    'divgen_initialization_qd_arrows',{'pink':'tab:orange','whitefloor':'tab:green'})
print('Arrow endpoints are prompt-equal means. Both conditions use their own saved baseline images.')
print('Times are per prompt and initialisation, for four images. Warm-start compute excludes loading and observer file export; prompt encoding is included in group setup.')
print('Intermediate budget costs are shared-trajectory prefixes, not separately measured standalone latencies.')
print('Paired LPIPS/DreamSim changes are in paired_change.csv; larger changes are not inherently better.')


# Complete warm end-to-end time is measured for the full trajectory only.
END_TO_END=TIMING[TIMING.evaluations==max(BUDGETS)].copy()
if not np.isfinite(END_TO_END.warm_start_end_to_end_seconds).all():
    raise RuntimeError('Missing complete warm end-to-end group timings.')
END_TO_END_SUMMARY=END_TO_END.groupby('init_name').warm_start_end_to_end_seconds.agg(['mean','median','min','max'])
END_TO_END_SUMMARY.to_csv(metrics_dir/'warm_end_to_end_timing_summary.csv')
print('Complete warm end-to-end group time, including observer export and final logging; model loading is separate.')
display(END_TO_END_SUMMARY)


## 14. Compare with quality-only FloorRefine

Set the two FloorRefine paths below to its completed generation and combined evaluation folders. The comparison checks the ordered prompts, seeds, generation settings and evaluator versions before forming paired score differences. It compares the whitefloor quality-only FloorRefine output with both DivGen initialisations. The optional overlay draws each method from its own saved baseline; the two whitefloor baselines are not merged because the generation precision differs. The other FloorRefine initialisations remain its internal ablation.

The saved FloorRefine optimisation timing excludes baseline generation and reward setup. DivGen's candidate loop includes the initial candidate. The cost table retains these boundaries explicitly, so it must not be presented as an end-to-end speedup. A formal runtime advantage requires matching timing boundaries. The Q–D comparison can use the existing images.

The default FloorRefine path uses the denoising step count of the selected DivGen results and a 10-update refinement budget. Adjust the path for another budget or run name. A one-step run cannot be paired with a four-step run in this comparison.


In [ ]:
# Optional: compare with the completed quality-only FloorRefine run; no image generation occurs here.
FLOORREFINE_RUN = PROJECT_ROOT/'denoising_optimization/outputs/initial_noise_optimize_geneval50'/f'geneval50_quality10_sdxl{EVALUATED_NUM_STEPS}step_v1'  # Adjust if using a different refinement budget or run name.
FLOORREFINE_METRICS = FLOORREFINE_RUN/'metrics_revalidated_v1/combined_with_maniqa'

def compare_floorrefine():
    floor = json.loads((FLOORREFINE_RUN/'manifest.json').read_text())
    if floor.get('status') != 'complete' or floor.get('method') != 'initial_noise_geneval50_three_init':
        raise ValueError('Select a complete FloorRefine generation run.')
    if floor.get('DIVERSITY_ENABLED') is not False or floor.get('QUALITY_WEIGHT', 0) <= 0:
        raise ValueError('This comparison requires quality-only FloorRefine.')
    for key, expected in [('MODEL_ID', 'stabilityai/sdxl-turbo'), ('VAE_ID', 'madebyollin/sdxl-vae-fp16-fix'),
                          ('HEIGHT', 512), ('WIDTH', 512), ('NUM_STEPS', EVALUATED_NUM_STEPS), ('EDIT_AFTER_STEP', 0)]:
        if floor.get(key) != expected: raise ValueError(f'FloorRefine generation setting differs: {key}')
    fields = ['prompt_id', 'prompt', 'category', 'batch_seed']
    signature = lambda prompts: [{key:p[key] for key in fields} for p in prompts]
    if len(prompt_ids) != 50 or signature(floor['prompts']) != signature(GENERATION_MANIFEST['prompts']):
        raise ValueError('External comparison requires the same ordered 50 prompts and batch seeds.')
    floor_meta = FLOORREFINE_METRICS.parent/'core/metric_manifest.json'
    previous = json.loads(floor_meta.read_text())
    current = json.loads((EVAL_ROOT/'core/metric_manifest.json').read_text())
    sources = json.loads((FLOORREFINE_METRICS/'evaluation_sources.json').read_text())
    if sources['generation_manifest_sha256'] != sha256_file(FLOORREFINE_RUN/'manifest.json'):
        raise RuntimeError('FloorRefine evaluation refers to another generation manifest.')
    if sha256_file(floor_meta) not in {s['sha256'] for s in sources['sources']}:
        raise RuntimeError('FloorRefine combined results do not reference the selected core evaluation.')
    checks = ['backend', 'metric_runner_sha256', 'worker_sha256', 'device', 'clip_checkpoint', 'lpips_resize', 'vendi_definition']
    mismatches = {key:(previous.get(key), current.get(key)) for key in checks if previous.get(key) != current.get(key)}
    packages = ['torch', 'torchvision', 'transformers', 'tokenizers', 'huggingface-hub', 'peft', 'trl',
                'numpy', 'Pillow', 'timm', 'lpips', 'dreamsim', 'vendi-score', 'hpsv3']
    for package in packages:
        a, b = previous['versions'].get(package), current['versions'].get(package)
        if a != b: mismatches[package] = (a, b)
    if mismatches:
        raise RuntimeError('Evaluation settings differ. Re-evaluate existing images consistently before comparison: '+str(mismatches))
    if RUN_MANIQA:
        if not sources.get('includes_maniqa'): raise ValueError('Add MANIQA to the FloorRefine evaluation, or set RUN_MANIQA=False.')
        maniqa_path = FLOORREFINE_METRICS.parent/'maniqa/metric_manifest.json'
        a = json.loads(maniqa_path.read_text())
        b = json.loads((EVAL_ROOT/'maniqa/metric_manifest.json').read_text())
        if sha256_file(maniqa_path) not in {s['sha256'] for s in sources['sources']}:
            raise RuntimeError('FloorRefine combined results reference another MANIQA evaluation.')
        for key in ['worker_sha256', 'maniqa_sampling', 'device']:
            if a.get(key) != b.get(key): raise RuntimeError(f'MANIQA evaluation differs: {key}')
        for package in ['torch', 'torchvision', 'pyiqa', 'timm', 'numpy', 'Pillow', 'transformers']:
            if a['versions'].get(package) != b['versions'].get(package):
                raise RuntimeError(f'MANIQA environment differs: {package}')
    floor_group = pd.read_csv(FLOORREFINE_METRICS/'per_group.csv')
    floor_lookup = {r['condition']:r for r in floor['records']}
    for column in ['prompt_id', 'init_name', 'local_condition']:
        floor_group[column] = floor_group.condition.map(lambda key:floor_lookup[key][column])
    floor_group = floor_group[(floor_group.init_name=='whitefloor') & floor_group.local_condition.isin(['baseline','edit_1'])].copy()
    comparison = []
    common_metrics=[metric for metric in KEY_METRICS if metric in set(GROUP.metric) and metric in set(floor_group.metric)
                    and (metric!='maniqa' or RUN_MANIQA)]
    for metric in common_metrics:
        f = floor_group[(floor_group.metric==metric)&(floor_group.local_condition=='edit_1')].set_index('prompt_id').score.reindex(prompt_ids)
        baseline = floor_group[(floor_group.metric==metric)&(floor_group.local_condition=='baseline')].set_index('prompt_id').score.reindex(prompt_ids)
        fm, fl, fh = paired_mean_ci(f.to_numpy())
        paired_mean_ci(baseline.to_numpy())
        for init_name in INIT_NAMES:
            for budget in BUDGETS:
                condition = f'budget_{budget:04d}'
                d = GROUP[(GROUP.metric==metric)&(GROUP.init_name==init_name)&(GROUP.local_condition==condition)].set_index('prompt_id').score.reindex(prompt_ids)
                dm, dl, dh = paired_mean_ci(d.to_numpy())
                difference, low, high = paired_mean_ci(f.to_numpy()-d.to_numpy())
                comparison.append(dict(metric=metric, divgen_init_name=init_name, divgen_evaluations=budget, n_prompts=50,
                    floorrefine_mean=fm, divgen_mean=dm, floorrefine_minus_divgen=difference, ci_low=low, ci_high=high))
    result = pd.DataFrame(comparison)
    result.to_csv(metrics_dir/'floorrefine_vs_divgen_paired.csv', index=False)
    display(result)
    floor_group['init_name']='floorrefine'
    plot_group=pd.concat([GROUP[GROUP.metric.isin(common_metrics)],floor_group[floor_group.metric.isin(common_metrics)]],ignore_index=True)
    labels={name:'DivGen: '+INIT_LABELS[name] for name in INIT_NAMES}
    labels['floorrefine']='FloorRefine: whitefloor + quality-only'
    selections={budget:{**{name:f'budget_{budget:04d}' for name in INIT_NAMES},'floorrefine':'edit_1'} for budget in BUDGETS}
    save_qd_arrow_figures(plot_group,labels,{name:'baseline' for name in labels},selections,
        'floorrefine_vs_divgen_qd_arrows',{'pink':'tab:orange','whitefloor':'tab:green','floorrefine':'tab:blue'})
    floor_cost = pd.read_csv(FLOORREFINE_RUN/'optimization_summary.csv')
    floor_cost = floor_cost[floor_cost.init_name=='whitefloor'].set_index('prompt_id').reindex(prompt_ids)
    if not np.isfinite(floor_cost.optimization_seconds).all(): raise RuntimeError('Incomplete FloorRefine timing records.')
    cost = [dict(method='FloorRefine', optimization_seconds=float(floor_cost.optimization_seconds.mean()),
        peak_cuda_allocated_gib=float(floor_cost.peak_cuda_allocated_gb.mean()),
        scope='saved optimisation phase; baseline generation and reward setup excluded')]
    for (init_name,budget), part in TIMING.groupby(['init_name','evaluations']):
        cost.append(dict(method=f'DivGen: {INIT_LABELS[init_name]} ({budget} evaluations)', optimization_seconds=float(part.optimization_seconds.mean()),
            peak_cuda_allocated_gib=float(part.peak_cuda_allocated_gib.mean()),
            scope='candidate evaluations and updates, including the initial candidate; model and prompt setup excluded'))
    pd.DataFrame(cost).to_csv(metrics_dir/'floorrefine_vs_divgen_recorded_cost.csv', index=False)
    display(pd.DataFrame(cost))
    print('These phase timings have different boundaries. They do not establish an end-to-end speedup.')
    print('Use a common timed start/end protocol for a formal runtime advantage claim; existing images need not be regenerated for Q-D evaluation.')
    return result

if len(prompt_ids)!=50:
    print('External comparison skipped for a pilot run. Generate all 50 prompts for the FloorRefine comparison.')
elif (FLOORREFINE_METRICS/'per_group.csv').is_file() and (FLOORREFINE_RUN/'manifest.json').is_file():
    FLOORREFINE_COMPARISON = compare_floorrefine()
else:
    print('FloorRefine comparison skipped. Set FLOORREFINE_RUN and FLOORREFINE_METRICS to the completed results.')


## Interpreting the comparison

Compare pink and whitefloor at each saved budget with prompt-level pairing. Q-D arrows start at each method's own saved baseline. Report the shared generation settings and reference HPS reduction explicitly. Each run saves its own baseline; verify image equivalence before combining baselines across methods. The run records the official optimisation phases and the complete warm end-to-end group time; loading is reported separately. Intermediate budgets are shared-trajectory prefixes. Cross-method runtime claims require matching timing boundaries with FloorRefine.

The selected upstream source is distributed under its original MIT licence, included below.

<details><summary>DivGen licence</summary>

```text
MIT License

Copyright (c) 2026 Anne Harrington, A. Sophia Koepke, Shyamgopal Karthik

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
```
</details>
